# UrbanPulse — Traffic Analytics & Congestion Prediction
**Proyek:** UrbanPulse (lanjutan Data Engineer) · **Kota:** Surabaya · **Sumber:** layer Gold & Silver (CSV)

| Bagian | Peran | Isi |
|---|---|---|
| **Part A** | Data Analyst | EDA, ranking ruas, peak hour, tren, anomali (Q1–Q10) |
| **Part B** | ML Engineer | B1 Weather Forecasting (suhu, hujan) dan B2 Traffic Congestion Prediction |

Part B memakai variabel dari Part A, jadi jalankan notebook dari atas (Run All).

# PART A — Data Analyst
### Traffic & Weather Analytics (Descriptive & Diagnostic)

## 1. Project Overview

**Tujuan:** menganalisis pola lalu lintas di 10 ruas jalan Surabaya, mengidentifikasi ruas dan waktu dengan congestion tinggi, serta mendeteksi lonjakan congestion.

**Pertanyaan analisis**

| ID | Pertanyaan | Metode | Tabel utama |
|---|---|---|---|
| Q1 | Ruas jalan mana yang memiliki rata-rata congestion ratio tertinggi? | Ranking & comparative analysis | gold |
| Q2 | Pada jam berapa congestion paling tinggi terjadi? | Temporal aggregation | gold |
| Q5 | Ruas jalan mana yang memiliki kecepatan paling stabil? | Std & CV | gold |
| Q6 | Bagaimana perubahan kondisi lalu lintas dari waktu ke waktu? | Time-series analysis | silver 30 menit |
| Q7 | Apakah congestion berbeda antararea Surabaya? | Area comparison | gold |
| Q8 | Apakah kecepatan berkaitan dengan waktu tempuh (`current_travel_time`)? | Correlation analysis | silver 30 menit |
| Q9 | Seberapa besar variasi congestion pada setiap ruas? | Distribution analysis | gold |
| Q10 | Apakah ada lonjakan congestion pada waktu tertentu? | Anomaly detection (Z-score, IQR) | silver 30 menit |

**Batasan analisis**
- Cakupan data pendek (±19 jam: 31 Okt 2026 21:00 s.d. 1 Nov 2026 16:30) dan seluruhnya **akhir pekan**.
- Tidak ada hujan pada periode traffic (`precipitation = 0`), sehingga analisis hari kerja vs akhir pekan dan cuaca vs kecepatan **tidak dilakukan**. Data cuaca hanya ditampilkan sebagai konteks.
- Setiap jam hanya muncul satu kali (tidak ada pengulangan hari), sehingga pola jam bersifat deskriptif, bukan pola harian yang tervalidasi.
- `congestion_ratio = 1 − current_speed / free_flow_speed` (dibatasi minimal 0).

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ── Konfigurasi ──────────────────────────────────────────────
TZ = "Asia/Jakarta"
Z_THRESH = 3.0          # ambang |z| untuk anomali
IQR_K = 1.5             # pengali IQR untuk fence
CONGESTED_GT = 0.0      # baris dianggap "macet" jika congestion ratio > nilai ini

# Letakkan 5 file CSV di folder ./data (atau folder yang sama dengan notebook)
CANDIDATES = [Path("data"), Path("."), Path("/mnt/user-data/uploads")]
DATA_DIR = next(p for p in CANDIDATES if (p / "gold_dim_location.csv").exists())
print("DATA_DIR:", DATA_DIR.resolve())

DATA_DIR: /content


## 2. Data Loading & Understanding
Layer yang dipakai: **Gold** (`dim_location`, `traffic_hourly`, `weather_traffic_hourly`) sebagai tabel utama, **Silver** (`traffic_30min`, `weather_hourly`) untuk resolusi 30 menit dan konteks cuaca.
Kolom `segment_coordinates` pada silver tidak dimuat (tidak dipakai, ukurannya besar).

In [2]:
def to_ts(s):
    """Parse timestamp ber-offset (+07) menjadi tz-aware Asia/Jakarta."""
    return pd.to_datetime(s, utc=True).dt.tz_convert(TZ)


dim = pd.read_csv(DATA_DIR / "gold_dim_location.csv")

gold_traffic = pd.read_csv(DATA_DIR / "gold_traffic_hourly.csv")
gold_traffic["hour_ts"] = to_ts(gold_traffic["hour_ts"])

gold_wt = pd.read_csv(DATA_DIR / "gold_weather_traffic_hourly.csv")
gold_wt["hour_ts"] = to_ts(gold_wt["hour_ts"])
gold_wt["obs_date"] = pd.to_datetime(gold_wt["obs_date"])

silver_cols = [
    "location_id", "observation_ts", "current_speed", "free_flow_speed",
    "current_travel_time", "free_flow_travel_time", "confidence",
    "road_closure", "frc", "ingestion_mode", "fetched_at",
]
silver_traffic = pd.read_csv(DATA_DIR / "silver_traffic_30min.csv", usecols=silver_cols)
silver_traffic["observation_ts"] = to_ts(silver_traffic["observation_ts"])
silver_traffic["fetched_at"] = to_ts(silver_traffic["fetched_at"])

silver_weather = pd.read_csv(DATA_DIR / "silver_weather_hourly.csv")
silver_weather["hour_ts"] = to_ts(silver_weather["hour_ts"])

tables = {
    "gold_dim_location": dim,
    "gold_traffic_hourly": gold_traffic,
    "gold_weather_traffic_hourly": gold_wt,
    "silver_traffic_30min": silver_traffic,
    "silver_weather_hourly": silver_weather,
}
pd.DataFrame(
    {name: {"baris": len(t), "kolom": t.shape[1]} for name, t in tables.items()}
).T

,baris,kolom
gold_dim_location,10,6
gold_traffic_hourly,300,8
gold_weather_traffic_hourly,300,18
silver_traffic_30min,560,11
silver_weather_hourly,7710,11


In [3]:
# Struktur & tipe data tiap tabel
for name, t in tables.items():
    print(f"\n=== {name}  {t.shape}")
    print(t.dtypes.to_string())


=== gold_dim_location  (10, 6)
location_id       object
location_name     object
area              object
latitude         float64
longitude        float64
updated_at        object

=== gold_traffic_hourly  (300, 8)
location_id                                   object
hour_ts                 datetime64[ns, Asia/Jakarta]
avg_speed                                    float64
min_speed                                      int64
avg_congestion_ratio                         float64
max_congestion_ratio                         float64
observation_count                              int64
computed_at                                   object

=== gold_weather_traffic_hourly  (300, 18)
location_id                                   object
hour_ts                 datetime64[ns, Asia/Jakarta]
obs_date                              datetime64[ns]
hour_of_day                                    int64
day_of_week                                    int64
is_weekend                                      bo

In [4]:
# Integrasi dim_location (nama ruas & area) ke tabel Gold dan Silver
dim_cols = ["location_id", "location_name", "area", "latitude", "longitude"]

df = gold_wt.merge(dim[dim_cols], on="location_id", how="left", validate="m:1")

sv = silver_traffic.merge(dim[dim_cols], on="location_id", how="left", validate="m:1")
sv["slot"] = sv["observation_ts"].dt.floor("30min")
sv["hour_of_day"] = sv["slot"].dt.hour
sv["congestion_ratio"] = (1 - sv["current_speed"] / sv["free_flow_speed"]).clip(lower=0)

assert df["location_name"].notna().all() and sv["location_name"].notna().all(), "ada location_id tanpa dim"
dim[["location_id", "location_name", "area"]]

,location_id,location_name,area
0,LOC01,Ahmad Yani,Selatan
1,LOC02,Darmo,Selatan
2,LOC03,Mayjen Sungkono,Barat
3,LOC04,HR Muhammad,Barat
4,LOC05,Basuki Rahmat,Pusat
5,LOC06,Tunjungan,Pusat
6,LOC07,Gubeng,Timur
7,LOC08,Kertajaya,Timur
8,LOC09,Kenjeran,Utara
9,LOC10,Rajawali,Utara


In [5]:
# Konteks cuaca pada periode traffic (tidak dianalisis lebih lanjut)
weather_cols = ["temperature", "humidity", "precipitation", "wind_speed", "cloud_cover"]
print(f"Periode gold: {df['hour_ts'].min()}  ->  {df['hour_ts'].max()}")
print(f"Jam dengan hujan (precipitation > 0): {(df['precipitation'] > 0).sum()} dari {len(df)} baris")
df[weather_cols].describe().T

Periode gold: 2026-10-31 21:00:00+07:00  ->  2026-11-02 05:00:00+07:00
Jam dengan hujan (precipitation > 0): 0 dari 300 baris


,count,mean,std,min,25%,50%,75%,max
temperature,300.0000,29.6417,3.3286,25.1000,26.5000,29.7000,33.0000,35.6000
humidity,300.0000,66.9000,18.4109,34.0000,48.0000,69.0000,84.0000,92.0000
precipitation,300.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
wind_speed,300.0000,6.9893,4.0436,1.0000,4.0000,5.6500,11.1000,16.3000
cloud_cover,300.0000,68.6567,31.2963,3.0000,46.0000,74.0000,97.0000,100.0000


## 3. Data Quality Assessment
Missing values, duplikat, konsistensi timestamp, validasi nilai, dan cakupan data.

In [6]:
# 3.1 Missing values
miss = pd.DataFrame({
    name: t.isna().sum() for name, t in tables.items()
}).fillna(0).astype(int)
miss = miss[(miss > 0).any(axis=1)]
print("Kolom dengan missing value:" if len(miss) else "Tidak ada missing value pada semua tabel.")
miss

Tidak ada missing value pada semua tabel.


,gold_dim_location,gold_traffic_hourly,gold_weather_traffic_hourly,silver_traffic_30min,silver_weather_hourly


In [7]:
# 3.2 Duplikat berdasarkan kunci
dup = pd.Series({
    "gold_traffic_hourly (location_id, hour_ts)": gold_traffic.duplicated(["location_id", "hour_ts"]).sum(),
    "gold_weather_traffic_hourly (location_id, hour_ts)": gold_wt.duplicated(["location_id", "hour_ts"]).sum(),
    "silver_traffic_30min (location_id, observation_ts)": silver_traffic.duplicated(["location_id", "observation_ts"]).sum(),
    "silver_weather_hourly (location_id, hour_ts)": silver_weather.duplicated(["location_id", "hour_ts"]).sum(),
}, name="baris_duplikat")
dup.to_frame()

,baris_duplikat
"gold_traffic_hourly (location_id, hour_ts)",0
"gold_weather_traffic_hourly (location_id, hour_ts)",0
"silver_traffic_30min (location_id, observation_ts)",0
"silver_weather_hourly (location_id, hour_ts)",0


In [8]:
# 3.3 Timestamp: rentang, interval, dan gap per ruas
def gap_report(frame, ts_col, step):
    rows = []
    for loc, g in frame.sort_values(ts_col).groupby("location_id"):
        d = g[ts_col].diff().dropna()
        rows.append({
            "location_id": loc,
            "n": len(g),
            "mulai": g[ts_col].min(),
            "akhir": g[ts_col].max(),
            "gap_>step": int((d > step).sum()),
            "gap_terbesar": d.max(),
        })
    return pd.DataFrame(rows)

print("Gold hourly (langkah 1 jam)")
display(gap_report(gold_wt, "hour_ts", pd.Timedelta("1h")))
print("Silver 30 menit (langkah 30 menit)")
display(gap_report(silver_traffic, "observation_ts", pd.Timedelta("30min")))

Gold hourly (langkah 1 jam)


,location_id,n,mulai,akhir,gap_>step,gap_terbesar
0,LOC01,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
1,LOC02,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
2,LOC03,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
3,LOC04,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
4,LOC05,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
5,LOC06,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
6,LOC07,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
7,LOC08,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
8,LOC09,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00
9,LOC10,30,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,1,0 days 04:00:00


Silver 30 menit (langkah 30 menit)


,location_id,n,mulai,akhir,gap_>step,gap_terbesar
0,LOC01,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
1,LOC02,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
2,LOC03,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
3,LOC04,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
4,LOC05,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
5,LOC06,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
6,LOC07,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
7,LOC08,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
8,LOC09,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00
9,LOC10,56,2026-10-31 21:00:00+07:00,2026-11-02 05:00:00+07:00,4,0 days 03:30:00


In [9]:
# 3.4 Validasi nilai & konsistensi antar-layer
chk = []

def add(name, ok, detail=""):
    chk.append({"cek": name, "lolos": bool(ok), "detail": detail})

add("speed gold >= 0", (df[["avg_speed", "min_speed"]] >= 0).all().all())
add("min_speed <= avg_speed", (df["min_speed"] <= df["avg_speed"] + 1e-9).all())
add("congestion gold dalam [0, 1]", df[["avg_congestion_ratio", "max_congestion_ratio"]].apply(lambda s: s.between(0, 1)).all().all())
add("avg_congestion <= max_congestion", (df["avg_congestion_ratio"] <= df["max_congestion_ratio"] + 1e-9).all())
add("observation_count >= 1", (df["observation_count"] >= 1).all(), f"nilai: {sorted(int(v) for v in df['observation_count'].unique())}")
add("silver current_speed >= 0", (sv["current_speed"] >= 0).all())
add("silver free_flow_speed > 0", (sv["free_flow_speed"] > 0).all())
add("silver current_speed <= free_flow_speed", (sv["current_speed"] <= sv["free_flow_speed"]).all(),
    f"{(sv['current_speed'] > sv['free_flow_speed']).sum()} baris melebihi free flow")
add("silver confidence dalam [0, 1]", sv["confidence"].between(0, 1).all(),
    f"min confidence = {sv['confidence'].min():.3f}")
add("tidak ada road_closure", (~sv["road_closure"].astype(bool)).all(),
    f"{sv['road_closure'].astype(bool).sum()} baris closure")
add("gold traffic == gold weather_traffic (avg_speed)",
    gold_traffic.merge(gold_wt, on=["location_id", "hour_ts"], suffixes=("_a", "_b"))
    .eval("abs(avg_speed_a - avg_speed_b) < 1e-9").all())
pd.DataFrame(chk)

,cek,lolos,detail
0,speed gold >= 0,True,
1,min_speed <= avg_speed,True,
2,"congestion gold dalam [0, 1]",True,
3,avg_congestion <= max_congestion,True,
4,observation_count >= 1,True,"nilai: [1, 2]"
5,silver current_speed >= 0,True,
6,silver free_flow_speed > 0,True,
7,silver current_speed <= free_flow_speed,True,0 baris melebihi free flow
8,"silver confidence dalam [0, 1]",True,min confidence = 0.521
9,tidak ada road_closure,True,0 baris closure


In [10]:
# Rekonsiliasi: congestion hasil hitung ulang dari silver vs gold (rata-rata per ruas-jam)
sv["hour_ts"] = sv["slot"].dt.floor("h")
recon = (sv.groupby(["location_id", "hour_ts"], as_index=False)["congestion_ratio"].mean()
           .merge(gold_wt[["location_id", "hour_ts", "avg_congestion_ratio"]], on=["location_id", "hour_ts"], how="inner"))
print(f"Pasangan ruas-jam yang cocok: {len(recon)}")
print(f"Selisih absolut maks : {(recon['congestion_ratio'] - recon['avg_congestion_ratio']).abs().max():.4f}")
print(f"Korelasi             : {recon[['congestion_ratio', 'avg_congestion_ratio']].corr().iloc[0, 1]:.4f}")

Pasangan ruas-jam yang cocok: 300
Selisih absolut maks : 0.0000
Korelasi             : 1.0000


In [11]:
# 3.5 Cakupan data
cov = df.groupby("location_name").agg(
    jam_terobservasi=("hour_ts", "nunique"),
    observasi_per_jam_rata2=("observation_count", "mean"),
    jam_dengan_1_observasi=("observation_count", lambda s: int((s == 1).sum())),
)
print("Tanggal (obs_date):", sorted(df["obs_date"].dt.strftime("%Y-%m-%d").unique()))
print("is_weekend:", df["is_weekend"].value_counts().to_dict())
print("Jam yang tercakup (hour_of_day):", sorted(int(h) for h in df["hour_of_day"].unique()))
missing_hours = sorted(set(range(24)) - set(int(h) for h in df["hour_of_day"].unique()))
print("Jam TIDAK tercakup:", missing_hours)
cov

Tanggal (obs_date): ['2026-10-31', '2026-11-01', '2026-11-02']
is_weekend: {True: 250, False: 50}
Jam yang tercakup (hour_of_day): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Jam TIDAK tercakup: []


,jam_terobservasi,observasi_per_jam_rata2,jam_dengan_1_observasi
location_name,,,
Ahmad Yani,30,1.8667,4
Basuki Rahmat,30,1.8667,4
Darmo,30,1.8667,4
Gubeng,30,1.8667,4
HR Muhammad,30,1.8667,4
Kenjeran,30,1.8667,4
Kertajaya,30,1.8667,4
Mayjen Sungkono,30,1.8667,4
Rajawali,30,1.8667,4


**Ringkasan kualitas data.** Kelima tabel tidak punya missing value maupun duplikat. Seluruh 11 validasi nilai lolos, dan congestion Silver yang dihitung ulang cocok dengan Gold pada 300 pasangan ruas-jam (selisih maksimum 0,0000). Kelemahan utama ada pada cakupan waktu: tiap ruas punya 30 jam unik (Gold) dan 56 slot (Silver) antara 31 Okt 21:00 dan 2 Nov 05:00, dengan celah terbesar 4 jam. Setiap jam muncul sekali per ruas, dan 250 dari 300 baris jatuh pada akhir pekan.

## 4. Exploratory Data Analysis
Distribusi congestion dan speed, serta karakteristik tiap ruas.

In [12]:
# 4.1 Statistik deskriptif (gold, level ruas-jam)
num = ["avg_speed", "min_speed", "avg_congestion_ratio", "max_congestion_ratio"]
desc = df[num].describe(percentiles=[.25, .5, .75, .9, .95]).T
desc["share_zero"] = [(df[c] == 0).mean() for c in num]
desc

,count,mean,std,min,25%,50%,75%,90%,95%,max,share_zero
avg_speed,300.0000,32.4817,3.7081,20.0000,30.0000,33.0000,34.0000,39.0000,40.0000,40.0000,0.0000
min_speed,300.0000,32.0700,4.0290,20.0000,30.0000,32.0000,34.0000,39.0000,40.0000,40.0000,0.0000
avg_congestion_ratio,300.0000,0.0278,0.0658,0.0000,0.0000,0.0000,0.0000,0.1154,0.1969,0.3505,0.8067
max_congestion_ratio,300.0000,0.0399,0.0865,0.0000,0.0000,0.0000,0.0000,0.2073,0.2334,0.3590,0.8067


In [13]:
# 4.2 Distribusi congestion & speed
fig = make_subplots(rows=1, cols=2, subplot_titles=("Distribusi avg_congestion_ratio", "Distribusi avg_speed (km/jam)"))
fig.add_trace(go.Histogram(x=df["avg_congestion_ratio"], nbinsx=30, name="congestion"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["avg_speed"], nbinsx=30, name="speed"), row=1, col=2)
fig.update_layout(showlegend=False, height=380, bargap=0.05)
fig.show()
print(f"Share congestion = 0 : {(df['avg_congestion_ratio'] == 0).mean():.1%}")
print(f"Skewness congestion  : {df['avg_congestion_ratio'].skew():.2f}")

Share congestion = 0 : 80.7%
Skewness congestion  : 2.54


In [14]:
# 4.3 Karakteristik ruas (silver 30 menit)
road = (sv.groupby(["location_id", "location_name", "area"])
          .agg(frc=("frc", lambda s: s.mode().iat[0]),
               free_flow_speed=("free_flow_speed", "median"),
               free_flow_travel_time=("free_flow_travel_time", "median"),
               speed_rata2=("current_speed", "mean"),
               observasi=("current_speed", "size"))
          .reset_index())
road

,location_id,location_name,area,frc,free_flow_speed,free_flow_travel_time,speed_rata2,observasi
0,LOC01,Ahmad Yani,Selatan,FRC2,40.0000,440.0000,38.6607,56
1,LOC02,Darmo,Selatan,FRC2,34.0000,547.0000,33.6250,56
2,LOC03,Mayjen Sungkono,Barat,FRC3,30.0000,472.0000,28.4286,56
3,LOC04,HR Muhammad,Barat,FRC3,39.0000,222.0000,36.2321,56
4,LOC05,Basuki Rahmat,Pusat,FRC2,34.0000,569.0000,32.3929,56
5,LOC06,Tunjungan,Pusat,FRC2,30.0000,243.0000,28.9107,56
6,LOC07,Gubeng,Timur,FRC2,31.0000,"1,032.0000",30.6964,56
7,LOC08,Kertajaya,Timur,FRC3,30.0000,"1,161.0000",29.8214,56
8,LOC09,Kenjeran,Utara,FRC1,33.0000,"1,639.0000",33.0000,56
9,LOC10,Rajawali,Utara,FRC1,33.0000,"1,639.0000",33.0000,56


**Insight EDA.**
- Congestion sangat miring: 80,7% baris ruas-jam bernilai 0, rata-rata 0,0278, maksimum 0,3505, skewness 2,54. Kecepatan rata-rata 32,48 km/jam (simpangan baku 3,71).
- Histogram kecepatan punya beberapa puncak (30, 33, 40 km/jam) yang mengikuti free-flow tiap ruas. Free-flow berkisar 30 sampai 40 km/jam, jadi perbandingan antarruas memakai congestion ratio, bukan kecepatan mentah.
- Kenjeran dan Rajawali (FRC1) berkecepatan rata-rata 33,00 km/jam, sama dengan free-flow.

## 5. Analysis
Catatan metodologi: rata-rata dihitung dari nilai per-jam (gold) tanpa pembobotan, kecuali kolom berlabel `_w` (dibobot `observation_count`).

### 5.1 Q1 — Ranking congestion ratio per ruas

In [15]:
def wavg(g, col):
    return np.average(g[col], weights=g["observation_count"])

q1 = (df.groupby(["location_id", "location_name", "area"])
        .apply(lambda g: pd.Series({
            "mean_congestion": g["avg_congestion_ratio"].mean(),
            "mean_congestion_w": wavg(g, "avg_congestion_ratio"),
            "median_congestion": g["avg_congestion_ratio"].median(),
            "max_congestion": g["max_congestion_ratio"].max(),
            "jam_macet": int((g["avg_congestion_ratio"] > CONGESTED_GT).sum()),
            "n_jam": len(g),
        }))
        .reset_index()
        .sort_values("mean_congestion", ascending=False)
        .reset_index(drop=True))
q1.index = q1.index + 1
q1.index.name = "rank"
display(q1)

fig = px.bar(q1.sort_values("mean_congestion"), x="mean_congestion", y="location_name",
             color="area", orientation="h", text_auto=".3f",
             title="Rata-rata congestion ratio per ruas")
fig.update_layout(height=450, yaxis_title=None, xaxis_title="rata-rata congestion ratio")
fig.show()

,location_id,location_name,area,mean_congestion,mean_congestion_w,median_congestion,max_congestion,jam_macet,n_jam
rank,,,,,,,,,
1,LOC04,HR Muhammad,Barat,0.0643,0.0689,0.0000,0.3590,11.0000,30.0000
2,LOC05,Basuki Rahmat,Pusat,0.0543,0.0541,0.0000,0.2857,11.0000,30.0000
3,LOC03,Mayjen Sungkono,Barat,0.0532,0.0570,0.0000,0.2903,9.0000,30.0000
4,LOC06,Tunjungan,Pusat,0.0382,0.0354,0.0000,0.3103,10.0000,30.0000
5,LOC01,Ahmad Yani,Selatan,0.0374,0.0356,0.0000,0.2750,7.0000,30.0000
6,LOC02,Darmo,Selatan,0.0137,0.0110,0.0000,0.2353,4.0000,30.0000
7,LOC07,Gubeng,Timur,0.0091,0.0098,0.0000,0.2258,3.0000,30.0000
8,LOC08,Kertajaya,Timur,0.0080,0.0086,0.0000,0.1562,3.0000,30.0000
9,LOC09,Kenjeran,Utara,0.0000,0.0000,0.0000,0.0000,0.0000,30.0000


**Insight Q1.** HR Muhammad (0,0643), Basuki Rahmat (0,0543), dan Mayjen Sungkono (0,0532) menempati tiga teratas dengan selisih kecil. Pada rata-rata berbobot `observation_count`, Mayjen Sungkono (0,0570) naik ke peringkat 2 di atas Basuki Rahmat (0,0541). Median semua ruas 0, jadi rata-rata ditentukan oleh 9 sampai 11 jam macet per ruas dari 30 jam. Kenjeran dan Rajawali tidak pernah macet.

### 5.2 Q2 — Congestion menurut jam
Cakupan jam: 21–23 (31 Okt) dan 0–16 (1 Nov). Jam 17–20 tidak ada di data.

In [16]:
q2 = (df.groupby("hour_of_day")
        .agg(mean_congestion=("avg_congestion_ratio", "mean"),
             max_congestion=("max_congestion_ratio", "max"),
             share_macet=("avg_congestion_ratio", lambda s: (s > CONGESTED_GT).mean()),
             ruas_macet=("avg_congestion_ratio", lambda s: int((s > CONGESTED_GT).sum())),
             n_obs=("avg_congestion_ratio", "size"))
        .reindex(range(24)))
display(q2.dropna(subset=["n_obs"]).sort_values("mean_congestion", ascending=False).head(5)
          .rename_axis("jam (top 5 by mean)"))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=q2.index, y=q2["mean_congestion"], name="rata-rata congestion")
fig.add_scatter(x=q2.index, y=q2["share_macet"], name=f"share ruas macet (> {CONGESTED_GT})",
                mode="lines+markers", secondary_y=True)
fig.update_layout(title="Congestion per jam (semua ruas)", height=400,
                  xaxis=dict(dtick=1, title="jam (WIB)"))
fig.update_yaxes(title_text="rata-rata congestion", secondary_y=False)
fig.update_yaxes(title_text="share macet", tickformat=".0%", secondary_y=True)
fig.show()

,mean_congestion,max_congestion,share_macet,ruas_macet,n_obs
jam (top 5 by mean),,,,,
15,0.0985,0.2571,0.7000,7,10
17,0.0914,0.3077,0.5000,5,10
18,0.0904,0.3077,0.6000,6,10
19,0.0843,0.3590,0.4000,4,10
13,0.0808,0.2903,0.7000,7,10


In [17]:
# Heatmap ruas x jam
hm = (df.pivot_table(index="location_name", columns="hour_of_day", values="avg_congestion_ratio", aggfunc="mean")
        .reindex(columns=range(24)))
hm = hm.loc[q1.set_index("location_name").index]   # urut sesuai ranking Q1
fig = px.imshow(hm, aspect="auto", color_continuous_scale="YlOrRd",
                labels=dict(x="jam (WIB)", y="ruas", color="congestion"),
                title="Heatmap congestion ruas x jam (kosong = tidak ada data)")
fig.update_xaxes(dtick=1)
fig.update_layout(height=450)
fig.show()

**Insight Q2.** Rata-rata tertinggi terjadi pada jam 15 (0,0985; 7 dari 10 ruas macet), disusul jam 17, 18, 19, dan 13. Semua jam 0 sampai 23 tercakup (n_obs = 10 per jam). Congestion nol pada jam 0 sampai 7, mulai naik pada jam 8 sampai 10, dan turun ke nol pada jam 23. Nilai maksimum tunggal 0,3590 terjadi pada jam 19, dan heatmap paling gelap pada HR Muhammad di jam 16 sampai 19. Tiap jam hanya diamati satu hari, jadi pola ini deskriptif.

### 5.3 Q5 — Stabilitas kecepatan per ruas
Stabil = variasi kecepatan kecil. CV (std / mean) membuat ruas dengan kecepatan rata-rata berbeda bisa dibandingkan; makin kecil makin stabil.

In [18]:
q5 = (df.groupby(["location_id", "location_name", "area"])["avg_speed"]
        .agg(mean_speed="mean", std_speed="std", min_speed="min", max_speed="max")
        .reset_index())
q5["range_speed"] = q5["max_speed"] - q5["min_speed"]
q5["cv"] = q5["std_speed"] / q5["mean_speed"]
q5 = q5.sort_values("cv").reset_index(drop=True)
q5.index = q5.index + 1
q5.index.name = "rank (1 = paling stabil)"
display(q5)

fig = px.bar(q5.sort_values("cv", ascending=False), x="cv", y="location_name", color="area",
             orientation="h", text_auto=".3f", title="Coefficient of variation kecepatan (lebih kecil = lebih stabil)")
fig.update_layout(height=450, yaxis_title=None)
fig.show()

,location_id,location_name,area,mean_speed,std_speed,min_speed,max_speed,range_speed,cv
rank (1 = paling stabil),,,,,,,,,
1,LOC10,Rajawali,Utara,33.0000,0.0000,33.0000,33.0000,0.0000,0.0000
2,LOC09,Kenjeran,Utara,33.0000,0.0000,33.0000,33.0000,0.0000,0.0000
3,LOC08,Kertajaya,Timur,29.8333,0.5142,28.0000,30.0000,2.0000,0.0172
4,LOC07,Gubeng,Timur,30.7167,0.8875,27.5000,31.0000,3.5000,0.0289
5,LOC02,Darmo,Selatan,33.5333,1.4794,27.0000,34.0000,7.0000,0.0441
6,LOC05,Basuki Rahmat,Pusat,32.3833,2.4236,27.0000,34.0000,7.0000,0.0748
7,LOC01,Ahmad Yani,Selatan,38.5833,2.9039,30.0000,40.0000,10.0000,0.0753
8,LOC06,Tunjungan,Pusat,28.8167,2.2571,20.0000,30.0000,10.0000,0.0783
9,LOC03,Mayjen Sungkono,Barat,28.5333,2.4668,22.5000,30.0000,7.5000,0.0865


**Insight Q5.** HR Muhammad paling tidak stabil (CV 0,1108; rentang 14 km/jam), disusul Mayjen Sungkono (0,0865) dan Tunjungan (0,0783). Rajawali dan Kenjeran berkecepatan 33 km/jam pada semua pengamatan (CV 0). Nilai yang tidak bergeser selama 30 jam, termasuk jam sibuk, perlu dikonfirmasi ke penyedia data sebelum dianggap kondisi jalan yang sebenarnya.

### 5.4 Q6 — Perubahan kondisi lalu lintas dari waktu ke waktu
Resolusi 30 menit (silver).

In [19]:
city = (sv.groupby("slot")
          .agg(congestion=("congestion_ratio", "mean"),
               speed=("current_speed", "mean"),
               n_ruas=("location_id", "nunique"))
          .sort_index())
city["congestion_roll"] = city["congestion"].rolling(3, center=True, min_periods=1).mean()
city["speed_roll"] = city["speed"].rolling(3, center=True, min_periods=1).mean()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=city.index, y=city["congestion"], name="congestion", mode="lines", opacity=0.35)
fig.add_scatter(x=city.index, y=city["congestion_roll"], name="congestion (rolling 3 slot)", mode="lines")
fig.add_scatter(x=city.index, y=city["speed_roll"], name="speed (rolling 3 slot)", mode="lines", secondary_y=True)
fig.update_layout(title="Tren rata-rata semua ruas (30 menit)", height=420, hovermode="x unified")
fig.update_yaxes(title_text="congestion ratio", secondary_y=False)
fig.update_yaxes(title_text="speed (km/jam)", secondary_y=True)
fig.show()

In [20]:
# Tren per ruas: heatmap congestion ruas x waktu, dan speed per ruas
pv = sv.pivot_table(index="location_name", columns="slot", values="congestion_ratio", aggfunc="mean")
pv = pv.loc[q1.set_index("location_name").index]
fig = px.imshow(pv, aspect="auto", color_continuous_scale="YlOrRd",
                labels=dict(x="waktu", y="ruas", color="congestion"),
                title="Congestion per ruas dari waktu ke waktu (30 menit)")
fig.update_layout(height=450)
fig.show()

fig = px.line(sv.sort_values("slot"), x="slot", y="current_speed", color="location_name",
              title="Kecepatan per ruas dari waktu ke waktu")
fig.update_layout(height=450, xaxis_title="waktu", yaxis_title="km/jam")
fig.show()

**Insight Q6.** Rata-rata kota hampir bebas congestion dari dini hari sampai sekitar pukul 08:00 pada 1 Nov, naik bertahap sejak pagi, mencapai puncak pada sore hari (rata-rata bergulir sekitar 0,09), lalu turun ke 0 menjelang tengah malam. Kecepatan kota bergerak berlawanan, dari sekitar 33 ke sedikit di bawah 31 km/jam. Selisih 2 sampai 3 km/jam itu kecil karena ruas dengan congestion nol tidak berubah. Heatmap per ruas menunjukkan congestion terkonsentrasi di HR Muhammad, Basuki Rahmat, Mayjen Sungkono, dan Tunjungan dari tengah hari sampai sekitar 20:00. Pada malam 31 Okt (sisi kiri heatmap) congestion muncul di Basuki Rahmat, Tunjungan, Ahmad Yani, dan Darmo.

### 5.5 Q7 — Perbandingan congestion antararea
Catatan: observasi per-jam dalam satu ruas tidak independen dan sampel pendek, sehingga uji statistik di bawah hanya indikatif.

In [21]:
q7 = (df.groupby("area")
        .agg(n_ruas=("location_id", "nunique"),
             mean_congestion=("avg_congestion_ratio", "mean"),
             median_congestion=("avg_congestion_ratio", "median"),
             std_congestion=("avg_congestion_ratio", "std"),
             max_congestion=("max_congestion_ratio", "max"),
             share_macet=("avg_congestion_ratio", lambda s: (s > CONGESTED_GT).mean()),
             mean_speed=("avg_speed", "mean"))
        .sort_values("mean_congestion", ascending=False))
display(q7)

fig = px.box(df, x="area", y="avg_congestion_ratio", color="area", points="all",
             category_orders={"area": q7.index.tolist()},
             title="Distribusi congestion per area")
fig.update_layout(height=420, showlegend=False)
fig.show()

groups = [g["avg_congestion_ratio"].values for _, g in df.groupby("area")]
try:
    h, p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis antararea: H = {h:.3f}, p = {p:.4f}")
except ValueError as e:
    print("Kruskal-Wallis tidak dapat dihitung:", e)

,n_ruas,mean_congestion,median_congestion,std_congestion,max_congestion,share_macet,mean_speed
area,,,,,,,
Barat,2,0.0587,0.0000,0.0948,0.3590,0.3333,32.4750
Pusat,2,0.0462,0.0000,0.0766,0.3103,0.3500,30.6000
Selatan,2,0.0256,0.0000,0.0625,0.2750,0.1833,36.0583
Timur,2,0.0086,0.0000,0.0268,0.2258,0.1000,30.2750
Utara,2,0.0000,0.0000,0.0000,0.0000,0.0000,33.0000


Kruskal-Wallis antararea: H = 35.951, p = 0.0000


**Insight Q7.** Barat punya rata-rata congestion tertinggi (0,0587), Pusat punya share jam macet tertinggi (35,0%), dan Utara nol. Kruskal-Wallis menunjukkan perbedaan antararea (H = 35,951; p < 0,0001). Tiap area hanya berisi 2 ruas dan observasi dalam satu ruas tidak independen, jadi hasil ini indikatif. Kecepatan rata-rata tidak mencerminkan kemacetan antararea: Selatan tercepat (36,06 km/jam) karena free-flow Ahmad Yani 40 km/jam, sedangkan free-flow Timur hanya 30 sampai 31 km/jam.

### 5.6 Q8 — Kecepatan vs `current_travel_time`
Waktu tempuh bergantung pada **panjang ruas** (berbeda tiap ruas), sehingga korelasi gabungan (pooled) bisa tercampur efek antarruas. Karena itu dihitung juga korelasi **di dalam tiap ruas**.

In [22]:
x, y = "current_speed", "current_travel_time"

pooled = pd.DataFrame({
    "metode": ["Pearson", "Spearman"],
    "r": [sv[[x, y]].corr("pearson").iloc[0, 1], sv[[x, y]].corr("spearman").iloc[0, 1]],
})
display(pooled)

def within(g):
    if g[x].nunique() < 2 or g[y].nunique() < 2:
        return pd.Series({"pearson": np.nan, "spearman": np.nan, "n": len(g)})
    return pd.Series({"pearson": g[x].corr(g[y]),
                      "spearman": g[x].corr(g[y], method="spearman"),
                      "n": len(g)})

q8 = sv.groupby("location_name").apply(within).sort_values("spearman")
display(q8)
print(f"Ruas tanpa variasi (korelasi tidak terdefinisi): {q8['pearson'].isna().sum()}")

fig = px.scatter(sv, x=x, y=y, color="location_name", opacity=0.7,
                 title="current_speed vs current_travel_time (silver 30 menit)")
fig.update_layout(height=450)
fig.show()

,metode,r
0,Pearson,-0.1368
1,Spearman,-0.1344


,pearson,spearman,n
location_name,,,
Ahmad Yani,-0.9986,-1.0000,56.0000
Basuki Rahmat,-0.9974,-1.0000,56.0000
Kertajaya,-0.9999,-1.0000,56.0000
HR Muhammad,-0.9962,-1.0000,56.0000
Mayjen Sungkono,-0.9985,-1.0000,56.0000
Tunjungan,-0.9941,-1.0000,56.0000
Gubeng,-0.9991,-1.0000,56.0000
Darmo,-0.9987,-1.0000,56.0000
Kenjeran,NaN,NaN,56.0000


Ruas tanpa variasi (korelasi tidak terdefinisi): 2


**Insight Q8.** Korelasi gabungan lemah (Pearson -0,1368; Spearman -0,1344) karena panjang ruas berbeda. Di dalam 8 ruas yang bervariasi, korelasi hampir sempurna (Pearson -0,9941 sampai -0,9999; Spearman -1,0). Waktu tempuh sama dengan panjang ruas dibagi kecepatan, jadi variabel ini tidak menambah informasi di luar kecepatan. Temuan ini mendukung penundaan Travel Time Prediction sampai data panjang ruas tersedia. Kenjeran dan Rajawali tidak bervariasi, sehingga korelasinya tidak terdefinisi.

### 5.7 Q9 — Variasi congestion per ruas

In [23]:
q9 = (df.groupby("location_name")["avg_congestion_ratio"]
        .agg(n="size", mean="mean", std="std", min="min",
             p50=lambda s: s.quantile(.5), p90=lambda s: s.quantile(.9),
             max="max", iqr=lambda s: s.quantile(.75) - s.quantile(.25),
             share_zero=lambda s: (s == 0).mean())
        .sort_values("std", ascending=False))
display(q9)

order = q9.index.tolist()
fig = px.box(df, x="location_name", y="avg_congestion_ratio", points="all",
             category_orders={"location_name": order},
             title="Sebaran congestion per ruas (urut std menurun)")
fig.update_layout(height=450, xaxis_title=None)
fig.show()

,n,mean,std,min,p50,p90,max,iqr,share_zero
location_name,,,,,,,,,
HR Muhammad,30,0.0643,0.1010,0.0000,0.0000,0.2282,0.3505,0.1129,0.6333
Mayjen Sungkono,30,0.0532,0.0896,0.0000,0.0000,0.2010,0.2742,0.0972,0.7000
Basuki Rahmat,30,0.0543,0.0807,0.0000,0.0000,0.1868,0.2286,0.1000,0.6333
Ahmad Yani,30,0.0374,0.0759,0.0000,0.0000,0.1402,0.2500,0.0000,0.7667
Tunjungan,30,0.0382,0.0727,0.0000,0.0000,0.1200,0.3103,0.0500,0.6667
Darmo,30,0.0137,0.0435,0.0000,0.0000,0.0324,0.2059,0.0000,0.8667
Gubeng,30,0.0091,0.0286,0.0000,0.0000,0.0065,0.1129,0.0000,0.9000
Kertajaya,30,0.0080,0.0253,0.0000,0.0000,0.0065,0.1104,0.0000,0.9000
Kenjeran,30,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000


**Insight Q9.** HR Muhammad (std 0,1010), Mayjen Sungkono (0,0896), dan Basuki Rahmat (0,0807) bervariasi paling besar, dengan IQR sekitar 0,10 dan 63 sampai 70% jam bernilai nol. Ahmad Yani punya std 0,0759 tetapi IQR 0: congestion di sana muncul sebagai lonjakan jarang di atas dasar nol. Pada Darmo, Gubeng, dan Kertajaya (87 sampai 90% jam bernilai nol) hampir seluruh variasi berasal dari beberapa titik ekstrem.

## 6. Anomaly Detection (Q10)
Data: silver 30 menit.
- **Speed:** Z-score global (|z| > 3).
- **Congestion:** Z-score per ruas (z > 3) dan IQR per ruas (nilai > Q3 + 1,5·IQR), hanya sisi atas (lonjakan).

Catatan: distribusi congestion sangat miring (mayoritas 0). Jika Q1 = Q3 = 0 pada suatu ruas, fence IQR = 0 dan semua nilai > 0 ikut ditandai; Z-score pada ruas tanpa variasi tidak terdefinisi. Z-score global pada speed juga ikut menangkap perbedaan karakter antarruas (free-flow berbeda), bukan hanya kejadian ekstrem.

In [24]:
# Speed: Z-score global
mu, sd = sv["current_speed"].mean(), sv["current_speed"].std()
sv["speed_z"] = (sv["current_speed"] - mu) / sd
sv["speed_anom"] = sv["speed_z"].abs() > Z_THRESH

# Congestion: per ruas
grp = sv.groupby("location_id")["congestion_ratio"]
sv["cong_z"] = (sv["congestion_ratio"] - grp.transform("mean")) / grp.transform("std").replace(0, np.nan)
q1v, q3v = grp.transform(lambda s: s.quantile(.25)), grp.transform(lambda s: s.quantile(.75))
sv["iqr_upper"] = q3v + IQR_K * (q3v - q1v)
sv["cong_anom_z"] = sv["cong_z"] > Z_THRESH
sv["cong_anom_iqr"] = sv["congestion_ratio"] > sv["iqr_upper"]
sv["cong_anom_both"] = sv["cong_anom_z"] & sv["cong_anom_iqr"]

iqr0 = (grp.quantile(.75) - grp.quantile(.25) == 0)
print(f"Speed global: mean = {mu:.2f}, std = {sd:.2f}")
print(f"Ruas dengan IQR congestion = 0: {int(iqr0.sum())} dari {len(iqr0)}")

summary = pd.DataFrame({
    "metode": ["Speed Z-score global", "Congestion Z-score per ruas", "Congestion IQR per ruas", "Congestion Z & IQR (keduanya)"],
    "n_anomali": [sv["speed_anom"].sum(), sv["cong_anom_z"].sum(), sv["cong_anom_iqr"].sum(), sv["cong_anom_both"].sum()],
    "dari": len(sv),
})
summary["persen"] = summary["n_anomali"] / summary["dari"]
summary

Speed global: mean = 32.48, std = 3.87
Ruas dengan IQR congestion = 0: 7 dari 10


,metode,n_anomali,dari,persen
0,Speed Z-score global,1,560,0.0018
1,Congestion Z-score per ruas,8,560,0.0143
2,Congestion IQR per ruas,34,560,0.0607
3,Congestion Z & IQR (keduanya),8,560,0.0143


In [25]:
# Anomali per ruas
per_road = (sv.groupby("location_name")
              .agg(speed_anom=("speed_anom", "sum"),
                   cong_anom_z=("cong_anom_z", "sum"),
                   cong_anom_iqr=("cong_anom_iqr", "sum"),
                   cong_anom_both=("cong_anom_both", "sum"),
                   max_congestion=("congestion_ratio", "max"))
              .sort_values("cong_anom_both", ascending=False))
display(per_road)

cols = ["slot", "location_name", "area", "current_speed", "free_flow_speed", "congestion_ratio", "cong_z"]
print("Lonjakan congestion (lolos Z-score dan IQR):")
display(sv.loc[sv["cong_anom_both"], cols].sort_values("congestion_ratio", ascending=False).reset_index(drop=True))
print("Anomali speed (|z| > 3):")
display(sv.loc[sv["speed_anom"], cols + ["speed_z"]].sort_values("speed_z").reset_index(drop=True))

,speed_anom,cong_anom_z,cong_anom_iqr,cong_anom_both,max_congestion
location_name,,,,,
Kertajaya,0,3,4,3,0.1562
Darmo,0,2,4,2,0.2353
Gubeng,0,2,3,2,0.2258
Tunjungan,1,1,12,1,0.3103
Basuki Rahmat,0,0,0,0,0.2857
Ahmad Yani,0,0,9,0,0.2750
Kenjeran,0,0,0,0,0.0000
HR Muhammad,0,0,0,0,0.3590
Mayjen Sungkono,0,0,2,0,0.2903


Lonjakan congestion (lolos Z-score dan IQR):


,slot,location_name,area,current_speed,free_flow_speed,congestion_ratio,cong_z
0,2026-10-31 21:00:00+07:00,Tunjungan,Pusat,20,29,0.3103,3.6549
1,2026-11-01 15:00:00+07:00,Darmo,Selatan,26,34,0.2353,5.0553
2,2026-11-01 19:30:00+07:00,Gubeng,Timur,24,31,0.2258,5.0729
3,2026-10-31 22:00:00+07:00,Darmo,Selatan,27,34,0.2059,4.3923
4,2026-11-01 15:30:00+07:00,Gubeng,Timur,25,31,0.1935,4.3153
5,2026-11-01 16:30:00+07:00,Kertajaya,Timur,27,32,0.1562,4.5569
6,2026-11-01 13:00:00+07:00,Kertajaya,Timur,27,31,0.1290,3.7172
7,2026-11-01 15:30:00+07:00,Kertajaya,Timur,27,31,0.1290,3.7172


Anomali speed (|z| > 3):


,slot,location_name,area,current_speed,free_flow_speed,congestion_ratio,cong_z,speed_z
0,2026-10-31 21:00:00+07:00,Tunjungan,Pusat,20,29,0.3103,3.6549,-3.2264


In [26]:
# Visualisasi: timeline congestion dengan penanda anomali
fig = px.scatter(sv.sort_values("slot"), x="slot", y="congestion_ratio", color="location_name",
                 opacity=0.45, title="Congestion 30 menit — lonjakan ditandai (simbol X)")
a = sv[sv["cong_anom_both"]]
fig.add_trace(go.Scatter(x=a["slot"], y=a["congestion_ratio"], mode="markers", name="lonjakan (Z & IQR)",
                         marker=dict(symbol="x", size=11, color="black"),
                         text=a["location_name"], hoverinfo="text+x+y"))
fig.update_layout(height=470, xaxis_title="waktu", yaxis_title="congestion ratio")
fig.show()

fig = px.histogram(sv, x="current_speed", color="speed_anom", nbins=30,
                   title="Distribusi current_speed (anomali Z-score global berwarna terpisah)")
fig.update_layout(height=380)
fig.show()

**Insight Q10.**
- IQR menandai 34 titik (6,07%), tetapi 7 dari 10 ruas punya IQR 0, sehingga setiap nilai di atas 0 pada ruas itu otomatis keluar dari fence. Z-score per ruas lebih selektif (8 titik, 1,43%), dan semuanya juga lolos IQR.
- Delapan lonjakan jatuh pada Kertajaya (3), Darmo (2), Gubeng (2), dan Tunjungan (1), yaitu ruas yang biasanya lancar. Enam di antaranya terjadi pada 1 Nov pukul 13:00 sampai 19:30, dan dua pada malam 31 Okt.
- HR Muhammad, Mayjen Sungkono, Basuki Rahmat, dan Ahmad Yani tidak punya anomali karena congestion tinggi sudah menjadi kondisi biasa di sana. HR Muhammad dengan nilai 0,3590 tidak ditandai sama sekali.
- Satu-satunya anomali kecepatan adalah Tunjungan pada 31 Okt 21:00 (20 km/jam, free-flow 29, z = -3,23), yang juga menjadi lonjakan congestion tertinggi (0,3103). Z-score global pada kecepatan ikut menangkap perbedaan karakter antarruas.

## 7. Key Findings & Recommendations

### Temuan
1. Data bersih: tidak ada missing value atau duplikat, seluruh validasi nilai lolos, dan agregat Silver cocok dengan Gold.
2. Congestion rendah dan terkonsentrasi. 80,7% baris ruas-jam bernilai 0, dan nilai tertinggi 0,3590 masih di level LOS B.
3. HR Muhammad (0,0643), Basuki Rahmat (0,0543), dan Mayjen Sungkono (0,0532) paling padat. HR Muhammad juga paling tidak stabil (CV 0,1108). Kenjeran dan Rajawali tidak menunjukkan congestion.
4. Congestion menumpuk pada jam 13 sampai 19, dengan puncak jam 15 (rata-rata 0,0985; 7 dari 10 ruas macet). Jam 0 sampai 7 bebas congestion.
5. Area Barat dan Pusat paling padat, Utara nol. Kruskal-Wallis menunjukkan perbedaan antararea (H = 35,951; p < 0,0001), dengan catatan hanya 2 ruas per area.
6. Di dalam ruas, kecepatan dan waktu tempuh berkorelasi hampir sempurna (r sekitar -0,99 sampai -1,0), jadi waktu tempuh tidak menambah informasi.
7. Delapan lonjakan congestion terdeteksi pada ruas yang biasanya lancar (Kertajaya, Darmo, Gubeng, Tunjungan), sebagian besar pada 1 Nov siang sampai sore.
8. Suhu satu jam ke depan dapat diprediksi dengan MAE 0,55 sampai 0,58 °C, 34 sampai 37% lebih baik dari Naive. Prediksi hujan menunjukkan sinyal (PR-AUC 0,879) tetapi berstatus eksperimen awal.
9. Definisi macet menurut LOS tidak punya kelas positif pada data ini. Model kemacetan memakai ambang sementara 0,15, dan training belum jalan karena train kurang 3 positif dari syarat 30.

### Interpretasi
Selama periode data, lalu lintas Surabaya umumnya lancar, dengan kemacetan ringan di koridor barat dan pusat pada siang sampai sore. Ruas dengan congestion tertinggi juga ruas dengan kecepatan paling berfluktuasi (HR Muhammad, Mayjen Sungkono), jadi kedua ukuran saling menguatkan. Lonjakan pada ruas lancar menunjukkan gangguan lokal yang berbeda dari kepadatan rutin di barat dan pusat. Pada pemodelan cuaca, kondisi jam ini menjelaskan sebagian besar kondisi jam berikutnya, sehingga model hanya menambah perbaikan moderat di atas baseline.

### Keterbatasan
- Cakupan sekitar 30 jam (31 Okt malam sampai 2 Nov dini hari) dengan celah hingga 4 jam. Tiap jam muncul sekali, jadi pola jam belum teruji sebagai pola harian.
- 250 dari 300 baris Gold jatuh pada akhir pekan, sehingga hari kerja tidak bisa dibandingkan.
- Tidak ada hujan pada periode traffic, jadi hubungan cuaca dan kecepatan tidak dianalisis.
- Distribusi congestion sangat miring (80,7% nol). Rata-rata mudah terdorong oleh sedikit jam macet, dan IQR kurang informatif pada ruas yang penuh nol.
- Hanya 2 ruas per area, jadi uji antararea bersifat indikatif.
- Kecepatan Kenjeran dan Rajawali konstan pada 33 km/jam, dan penyebabnya belum dikonfirmasi.
- Ambang macet belum divalidasi, dan sumber data cuaca belum dikonfirmasi (kemungkinan reanalisis).
- Metrik hujan berasal dari 5 episode di test, dengan prevalensi yang berbeda jauh antarsplit (6,6%; 23,6%; 30,4%).
- Model kemacetan belum dilatih, jadi tujuan utama proyek belum punya ukuran akurasi.

### Rekomendasi
1. **Prioritas pemantauan:** HR Muhammad, Basuki Rahmat, dan Mayjen Sungkono pada pukul 13:00 sampai 20:00. Tunjungan dan Ahmad Yani menyusul sebagai ruas dengan lonjakan sesekali.
2. **Perpanjang pengumpulan data** minimal 2 sampai 4 minggu dengan hari kerja dan hari hujan, agar pola harian, perbedaan hari kerja dan akhir pekan, serta pengaruh cuaca dapat diuji.
3. **Validasi ambang macet** dengan acuan lokal sebelum hasil model dipakai untuk keputusan. Sampai saat itu, sebut label 0,15 sebagai "perlambatan".
4. **Periksa data Kenjeran dan Rajawali** ke penyedia data untuk memastikan kecepatan konstan 33 km/jam mencerminkan kondisi nyata.
5. **Tambah atribut ruas:** panjang ruas untuk Travel Time Prediction, dan lebih banyak ruas per area untuk perbandingan antararea.
6. **Konfirmasi sumber cuaca** (reanalisis atau observasi) sebelum model cuaca dipakai operasional.
7. **Jalankan B2** setelah syarat positif terpenuhi, dengan validasi berbasis waktu. Untuk hujan, kumpulkan lebih banyak episode, pilih ambang berdasarkan biaya alarm palsu versus hujan terlewat, dan laporkan PR-AUC bersama precision dan recall.

---
---
# PART B — ML Engineer
### Predictive Analytics · Supervised Learning

| Eksperimen | Fokus | Status data |
|---|---|---|
| **B1** | Weather Forecasting: suhu (regresi) dan kejadian hujan (klasifikasi biner), 1 jam ke depan | Bisa dikerjakan sekarang (761 jam) |
| **B2** | Traffic Congestion Prediction: macet / tidak macet, 30 menit ke depan | Pipeline siap, training menunggu data traffic lebih panjang |

> **Jalankan Part A dahulu** (Part B memakai `sv`, `silver_weather`, dan `dim` dari Part A).

## B1. Weather Forecasting

### B1.1 Project Overview

**Tujuan:** memprediksi kondisi cuaca Surabaya satu jam ke depan dari pola cuaca historis per jam.

**Pertanyaan ML**
1. **ML-Q1:** Bisakah suhu satu jam ke depan diprediksi? (regresi)
2. **ML-Q2:** Bisakah kejadian hujan satu jam ke depan diprediksi? (klasifikasi biner)
3. **ML-Q3:** Model mana yang terbaik dibanding baseline?
4. **ML-Q4:** Fitur cuaca apa yang paling berpengaruh?

**Keputusan scope:** target = suhu dan kejadian hujan; kelembapan ditunda. Model: Naive, Linear Regression, Random Forest, HistGradientBoosting (scikit-learn).

**Prinsip metodologi**
- **Unit analisis = jam unik (`hour_ts`).** Data punya 10 ruas per jam, tetapi cuaca antarruas nyaris identik (lihat B1.2), sehingga satu jam = satu kondisi cuaca kota. Seluruh ruas pada jam yang sama otomatis berada di split yang sama, dan baseline serta evaluasi dihitung pada jam unik, bukan 10 baris kembar.
- **Chronological split** 70 / 15 / 15 (train / validation / test) dengan purge satu jam di tiap batas. Hyperparameter dipilih pada validation; model final dilatih ulang pada train + validation dan dievaluasi sekali pada test.
- Fitur pada jam *t* hanya memakai data ≤ *t*; target adalah nilai pada *t + 1 jam*.
- Hujan: precision, recall, F1, PR-AUC, dan confusion matrix (bukan hanya accuracy). Hasil hujan berstatus **eksperimen awal**.

In [27]:
import itertools
import joblib
from sklearn.base import clone
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             mean_absolute_error, mean_squared_error, precision_score,
                             r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ── Konfigurasi B1 ───────────────────────────────────────────
W_RS = 42
W_TRAIN_FRAC, W_VAL_FRAC = 0.70, 0.15      # sisanya = test
MIN_RAIN_POS_TRAIN, MIN_RAIN_POS_VAL, MIN_RAIN_POS_TEST = 20, 5, 10

### B1.2 Data & Kualitas
**Sumber:** `silver_weather_hourly` (10 ruas × jam, 1 Okt – 1 Nov 2026). Sebagian besar baris bermode `backfill` dan sebagian kecil `live`.
Sumber API **belum dikonfirmasi** dari kode ingestion: diduga Open-Meteo Historical Weather API (reanalysis ERA5/ERA5-Land), yang berarti data adalah reanalysis, bukan observasi stasiun. Catatan ini perlu diperbarui setelah dikonfirmasi.

In [28]:
# 2.1 Kelengkapan: jumlah ruas per jam, dan mode ingestion
cnt = silver_weather.groupby("hour_ts").size()
print(f"Jam unik: {len(cnt)} | ruas per jam: {cnt.value_counts().to_dict()}")
print("Mode ingestion (baris):", silver_weather["ingestion_mode"].value_counts().to_dict())

# 2.2 Deret cuaca kota per jam unik
wx = (silver_weather.groupby("hour_ts")
        .agg(temperature=("temperature", "mean"),
             humidity=("humidity", "mean"),
             wind_speed=("wind_speed", "mean"),
             cloud_cover=("cloud_cover", "mean"),
             precip_mean=("precipitation", "mean"),
             precip_max=("precipitation", "max"),
             share_live=("ingestion_mode", lambda s: (s == "live").mean()))
        .sort_index())
full_idx = pd.date_range(wx.index.min(), wx.index.max(), freq="h", name="hour_ts")
print(f"Jam hilang pada rentang: {len(full_idx.difference(wx.index))}")
wx = wx.reindex(full_idx)

# 2.3 Seberapa mirip antarruas? (alasan satu jam = satu kondisi)
spread = silver_weather.groupby("hour_ts")[["temperature", "humidity", "wind_speed", "cloud_cover"]].std().mean()
print("\nRata-rata std antarruas pada jam yang sama:")
print(spread.round(3).to_string())

print(f"\nPeriode: {wx.index.min()} -> {wx.index.max()}  ({len(wx)} jam)")
print(f"Jam bermode live (sebagian/seluruhnya): {int((wx['share_live'] > 0).sum())}")
wx.describe().T

Jam unik: 771 | ruas per jam: {10: 771}
Mode ingestion (baris): {'backfill': 7411, 'live': 299}
Jam hilang pada rentang: 3

Rata-rata std antarruas pada jam yang sama:
temperature   0.4510
humidity      2.8450
wind_speed    1.5530
cloud_cover   2.9210

Periode: 2026-10-01 00:00:00+07:00 -> 2026-11-02 05:00:00+07:00  (774 jam)
Jam bermode live (sebagian/seluruhnya): 30


,count,mean,std,min,25%,50%,75%,max
temperature,771.0000,30.1420,2.9542,24.6600,27.7500,29.9000,32.8550,36.0900
humidity,771.0000,61.3983,16.5482,23.0000,46.5500,63.5000,74.1000,96.9000
wind_speed,771.0000,8.8164,4.5408,0.8700,4.6600,8.8700,12.6550,21.1500
cloud_cover,771.0000,52.1716,39.7090,0.0000,8.2500,53.4000,94.9000,100.0000
precip_mean,771.0000,0.0233,0.1171,0.0000,0.0000,0.0000,0.0000,1.6100
precip_max,771.0000,0.0506,0.2255,0.0000,0.0000,0.0000,0.0000,2.3000
share_live,771.0000,0.0388,0.1929,0.0000,0.0000,0.0000,0.0000,1.0000


In [29]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Suhu rata-rata kota (°C)", "Curah hujan maksimum antarruas (mm/jam)"))
fig.add_scatter(x=wx.index, y=wx["temperature"], mode="lines", name="suhu", row=1, col=1)
fig.add_bar(x=wx.index, y=wx["precip_max"], name="hujan (maks)", row=2, col=1)
live = wx.index[wx["share_live"] > 0]
if len(live):
    fig.add_vrect(x0=live.min(), x1=live.max(), fillcolor="gray", opacity=0.15, line_width=0,
                  annotation_text="mode live", row="all", col=1)
fig.update_layout(height=520, showlegend=False)
fig.show()

**Insight B1.2.** Data memuat 771 jam unik (10 ruas per jam, 3 jam hilang dalam rentang 774 jam). Simpangan baku antarruas pada jam yang sama kecil (suhu 0,45 °C, kelembapan 2,85, angin 1,55, tutupan awan 2,92), jadi cuaca kota diperlakukan sebagai satu deret dan sampel efektif jauh di bawah 7.710 baris. Suhu rata-rata 30,14 °C (24,66 sampai 36,09) dengan siklus harian yang jelas. Hanya 30 jam berstatus `live`, sisanya `backfill`.

### B1.3 Definisi Target Hujan
Target biner: **ada hujan terukur pada jam berikutnya** di kota (dihitung dari curah hujan maksimum antarruas pada jam itu).
Resolusi data 0,1 mm/jam, sehingga `> 0` berarti hujan terukur (≥ 0,1 mm). Kandidat ambang yang lebih tinggi ditampilkan sebagai pembanding; ambang yang dipakai tetap dapat diganti.

In [30]:
RAIN_CANDIDATES = [("> 0 (hujan terukur, >= 0,1 mm)", 0.0, ">"),
                   (">= 0.5 mm", 0.5, ">="), (">= 1.0 mm", 1.0, ">="), (">= 2.0 mm", 2.0, ">=")]

def w_flag(series, thr, op):
    s = series.round(6)
    return (s > thr) if op == ">" else (s >= thr)

_n_tail = int(len(wx) * W_VAL_FRAC)
rows = []
for name, thr, op in RAIN_CANDIDATES:
    f = w_flag(wx["precip_max"], thr, op)
    runs = int((f & ~f.shift(1, fill_value=False)).sum())
    rows.append({"ambang": name, "jam_hujan": int(f.sum()), "persen": f.mean(),
                 "episode_hujan": runs, "hari_dengan_hujan": int(f.groupby(wx.index.date).any().sum()),
                 "jam_hujan_di_15%_akhir": int(f.iloc[-_n_tail:].sum())})
display(pd.DataFrame(rows))

# ambang yang dipakai
RAIN_THRESHOLD, RAIN_OP = 0.0, ">"
rain_series = w_flag(wx["precip_max"], RAIN_THRESHOLD, RAIN_OP)
daily = rain_series.groupby(wx.index.date).sum()
fig = px.bar(x=[str(d) for d in daily.index], y=daily.values, labels={"x": "tanggal", "y": "jam hujan"},
             title="Jam hujan per hari (hujan mengelompok pada beberapa episode)")
fig.update_layout(height=340)
fig.show()

,ambang,jam_hujan,persen,episode_hujan,hari_dengan_hujan,jam_hujan_di_15%_akhir
0,"> 0 (hujan terukur, >= 0,1 mm)",94,0.1214,20,14,33
1,>= 0.5 mm,26,0.0336,12,8,10
2,>= 1.0 mm,13,0.0168,8,5,6
3,>= 2.0 mm,3,0.0039,3,2,2


**Insight B1.3.** Ambang > 0 mm menghasilkan 94 jam hujan (12,1%) dalam 20 episode dan 14 hari berhujan, dengan 33 jam di antaranya pada 15% waktu terakhir. Ambang ≥ 0,5 mm menyisakan 26 jam (12 episode), ≥ 1,0 mm 13 jam (8 episode), dan ≥ 2,0 mm 3 jam (3 episode). Ambang yang lebih tinggi menyisakan terlalu sedikit kejadian untuk dimodelkan, jadi notebook memakai > 0. Hujan mengelompok pada dua periode: awal Oktober (sekitar 4 sampai 12 Okt) dan akhir Oktober sampai 1 Nov.

### B1.4 Feature Engineering
Pada jam *t*, fitur memakai data ≤ *t*; target = nilai pada *t + 1 jam*.

| Kelompok | Fitur |
|---|---|
| Suhu | `temp_t`, `temp_lag_1/2/3/6`, `temp_yday_target` (suhu pada jam yang sama dengan jam target, kemarin), `temp_diff_1`, `temp_roll3_mean`, `temp_roll6_mean`, `temp_roll6_std` |
| Variabel lain | `humidity_t/lag_1`, `cloud_t/lag_1`, `wind_t/lag_1` |
| Hujan | `precip_t`, `precip_lag_1`, `precip_roll3_sum`, `rain_now`, `rain_hours_6h`, `rain_hours_24h` |
| Waktu (jam target) | `hour_sin`, `hour_cos` |

`day_of_week` dan `month` tidak dipakai: data hanya satu bulan sehingga tidak informatif dan berisiko overfit.

In [31]:
def make_weather_features(wx):
    d = wx.copy()
    d["is_rain"] = np.where(d["precip_max"].isna(), np.nan,
                            w_flag(d["precip_max"], RAIN_THRESHOLD, RAIN_OP).astype(float))
    t = d["temperature"]
    f = pd.DataFrame(index=d.index)
    f["temp_t"] = t
    for k in (1, 2, 3, 6):
        f[f"temp_lag_{k}"] = t.shift(k)
    f["temp_yday_target"] = t.shift(23)            # jam target (t+1) kemarin = t-23
    f["temp_diff_1"] = t - t.shift(1)
    f["temp_roll3_mean"] = t.rolling(3).mean()
    f["temp_roll6_mean"] = t.rolling(6).mean()
    f["temp_roll6_std"] = t.rolling(6).std()
    for col, short in (("humidity", "humidity"), ("cloud_cover", "cloud"), ("wind_speed", "wind")):
        f[f"{short}_t"] = d[col]
        f[f"{short}_lag_1"] = d[col].shift(1)
    f["precip_t"] = d["precip_mean"]
    f["precip_lag_1"] = d["precip_mean"].shift(1)
    f["precip_roll3_sum"] = d["precip_mean"].rolling(3).sum()
    f["rain_now"] = d["is_rain"]
    f["rain_hours_6h"] = d["is_rain"].rolling(6).sum()
    f["rain_hours_24h"] = d["is_rain"].rolling(24).sum()
    target_hour = (d.index + pd.Timedelta("1h")).hour
    f["hour_sin"] = np.sin(2 * np.pi * target_hour / 24)
    f["hour_cos"] = np.cos(2 * np.pi * target_hour / 24)
    f["y_temp"] = t.shift(-1)
    f["y_rain"] = d["is_rain"].shift(-1)
    return f

wf = make_weather_features(wx)
W_TARGETS = ["y_temp", "y_rain"]
W_FEATURES = [c for c in wf.columns if c not in W_TARGETS]
wm = wf.dropna().copy()
wm["y_rain"] = wm["y_rain"].astype(int)
assert wm.index.is_unique, "jam harus unik"
print(f"Jam siap dimodelkan: {len(wm)} dari {len(wx)} (sisanya hilang karena lag/rolling 24 jam & target t+1)")
print(f"Jumlah fitur: {len(W_FEATURES)}")
wm[W_FEATURES + W_TARGETS].head(3)

Jam siap dimodelkan: 742 dari 774 (sisanya hilang karena lag/rolling 24 jam & target t+1)
Jumlah fitur: 24


,temp_t,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_6,temp_yday_target,temp_diff_1,temp_roll3_mean,temp_roll6_mean,temp_roll6_std,humidity_t,humidity_lag_1,cloud_t,cloud_lag_1,wind_t,wind_lag_1,precip_t,precip_lag_1,precip_roll3_sum,rain_now,rain_hours_6h,rain_hours_24h,hour_sin,hour_cos,y_temp,y_rain
hour_ts,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-10-01 23:00:00+07:00,28.3100,28.7100,29.0700,29.1700,31.3000,27.9600,-0.4000,28.6967,29.1700,0.6308,71.7000,68.9000,38.1000,41.1000,12.4800,14.3900,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,27.9900,0
2026-10-02 00:00:00+07:00,27.9900,28.3100,28.7100,29.0700,29.9800,27.7300,-0.3200,28.3367,28.8383,0.6428,72.9000,71.7000,74.3000,38.1000,10.8200,12.4800,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2588,0.9659,27.6100,0
2026-10-02 01:00:00+07:00,27.6100,27.9900,28.3100,28.7100,29.7800,27.3000,-0.3800,27.9700,28.4767,0.6169,74.4000,72.9000,87.1000,74.3000,9.7800,10.8200,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.8660,27.3000,0


### B1.5 Temporal Split (per jam unik)
Urutan kronologis 70 / 15 / 15. Baris terakhir tiap segmen dibuang (*purge*) karena target-nya (t + 1 jam) jatuh di segmen berikutnya.

In [32]:
n = len(wm)
i_tr, i_va = int(n * W_TRAIN_FRAC), int(n * (W_TRAIN_FRAC + W_VAL_FRAC))
wtrain = wm.iloc[: i_tr - 1]
wval = wm.iloc[i_tr: i_va - 1]
wtest = wm.iloc[i_va:]

# validasi anti-leakage
assert wtrain.index.max() + pd.Timedelta("1h") < wval.index.min()
assert wval.index.max() + pd.Timedelta("1h") < wtest.index.min()
assert not (set(wtrain.index) & set(wval.index)) and not (set(wval.index) & set(wtest.index))
assert not (set(wtrain.index) & set(wtest.index))
# semua ruas pada jam yang sama ada di split yang sama (data sudah satu baris per jam)
assert wm.index.is_unique

def runs(s):
    return int(((s == 1) & (s.shift(1, fill_value=0) == 0)).sum())

split = pd.DataFrame({
    "jam": [len(wtrain), len(wval), len(wtest)],
    "mulai": [wtrain.index.min(), wval.index.min(), wtest.index.min()],
    "akhir": [wtrain.index.max(), wval.index.max(), wtest.index.max()],
    "suhu_rata2": [d["y_temp"].mean() for d in (wtrain, wval, wtest)],
    "suhu_std": [d["y_temp"].std() for d in (wtrain, wval, wtest)],
    "jam_hujan": [int(d["y_rain"].sum()) for d in (wtrain, wval, wtest)],
    "episode_hujan": [runs(d["y_rain"]) for d in (wtrain, wval, wtest)],
    "prevalensi_hujan": [d["y_rain"].mean() for d in (wtrain, wval, wtest)],
}, index=["train", "val", "test"])
display(split)

RAIN_READY = (int(wtrain["y_rain"].sum()) >= MIN_RAIN_POS_TRAIN and int(wval["y_rain"].sum()) >= MIN_RAIN_POS_VAL
              and int(wtest["y_rain"].sum()) >= MIN_RAIN_POS_TEST)
print("RAIN_READY =", RAIN_READY)

,jam,mulai,akhir,suhu_rata2,suhu_std,jam_hujan,episode_hujan,prevalensi_hujan
train,518,2026-10-01 23:00:00+07:00,2026-10-23 12:00:00+07:00,30.2442,2.8854,34,9,0.0656
val,110,2026-10-23 14:00:00+07:00,2026-10-28 03:00:00+07:00,29.8197,2.8915,26,6,0.2364
test,112,2026-10-28 05:00:00+07:00,2026-11-01 20:00:00+07:00,30.1682,3.2378,34,5,0.3036


RAIN_READY = True


**Insight B1.5.** Split menghasilkan train 518 jam, validasi 110 jam, dan test 112 jam. Prevalensi hujan berbeda jauh: 6,6% di train, 23,6% di validasi, dan 30,4% di test (34 jam hujan dalam 5 episode). Model hujan berlatih pada periode yang hampir kering dan diuji pada periode paling basah, sehingga metrik hujan sulit dibaca sebagai kinerja umum.

### B1.6 Eksperimen 1 — Prediksi Suhu 1 Jam ke Depan (ML-Q1, ML-Q3)
**Baseline:** *Naive* (suhu jam ini), *Seasonal Naive* (suhu pada jam target kemarin).
**Model:** Linear Regression, Random Forest, HistGradientBoosting; hyperparameter dipilih pada validation (MAE), lalu dilatih ulang pada train + validation.
**Metrik:** MAE, RMSE, R² pada test (jam unik).

In [33]:
def w_reg(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))), "R2": r2_score(y, p)}

def w_pipe(est):
    return Pipeline([("sc", StandardScaler()), ("m", est)])

def w_select(make_est, grid, Xtr, ytr, Xva, yva, score_fn, higher_better, predict):
    best_p, best_s = None, None
    for params in ParameterGrid(grid):
        m = w_pipe(make_est(**params)).fit(Xtr, ytr)
        s = score_fn(yva, predict(m, Xva))
        if best_s is None or (s > best_s if higher_better else s < best_s) or (np.isnan(best_s) and not np.isnan(s)):
            best_p, best_s = params, s
    return best_p, best_s

W_REG_SPECS = {
    "Linear Regression": (lambda **p: LinearRegression(**p), {}),
    "Random Forest": (lambda **p: RandomForestRegressor(n_estimators=300, random_state=W_RS, n_jobs=-1, **p),
                      {"max_depth": [4, 8, None], "min_samples_leaf": [1, 5]}),
    "HistGradientBoosting": (lambda **p: HistGradientBoostingRegressor(random_state=W_RS, **p),
                             {"learning_rate": [0.05, 0.1], "max_depth": [2, 3], "max_iter": [100, 200]}),
}

Xtr, ytr = wtrain[W_FEATURES], wtrain["y_temp"]
Xva, yva = wval[W_FEATURES], wval["y_temp"]
Xte, yte = wtest[W_FEATURES], wtest["y_temp"]
Xfit = pd.concat([Xtr, Xva]); yfit = pd.concat([ytr, yva])

temp_rows, temp_models, temp_pred = [], {}, {}
for name, base in (("Baseline: Naive (suhu jam ini)", wtest["temp_t"]),
                   ("Baseline: Seasonal Naive (jam target kemarin)", wtest["temp_yday_target"])):
    temp_rows.append({"model": name, **w_reg(yte, base), "val_MAE": np.nan, "best_params": ""})
    temp_pred[name] = base.values

for name, (mk, grid) in W_REG_SPECS.items():
    params, val_mae = w_select(mk, grid, Xtr, ytr, Xva, yva, mean_absolute_error, False, lambda m, X: m.predict(X))
    model = w_pipe(mk(**params)).fit(Xfit, yfit)
    p = model.predict(Xte)
    temp_models[name], temp_pred[name] = model, p
    temp_rows.append({"model": name, **w_reg(yte, p), "val_MAE": val_mae, "best_params": params})

temp_cmp = pd.DataFrame(temp_rows).set_index("model")
naive_mae = temp_cmp.loc["Baseline: Naive (suhu jam ini)", "MAE"]
temp_cmp["skill_vs_naive"] = 1 - temp_cmp["MAE"] / naive_mae
display(temp_cmp)

model_names = [m for m in temp_cmp.index if not m.startswith("Baseline")]
temp_best = min(model_names, key=lambda m: temp_cmp.loc[m, "val_MAE"])
print(f"Model terbaik berdasarkan MAE validation: {temp_best}")
print(f"(skill_vs_naive > 0 berarti lebih baik dari Naive; test = {len(wtest)} jam unik)")

,MAE,RMSE,R2,val_MAE,best_params,skill_vs_naive
model,,,,,,
Baseline: Naive (suhu jam ini),0.8768,1.1057,0.8823,NaN,,0.0000
Baseline: Seasonal Naive (jam target kemarin),0.9490,1.2274,0.8550,NaN,,-0.0824
Linear Regression,0.5541,0.7268,0.9492,0.6422,{},0.3680
Random Forest,0.5776,0.8053,0.9376,0.6252,"{'max_depth': None, 'min_samples_leaf': 1}",0.3413
HistGradientBoosting,0.5765,0.7466,0.9464,0.5130,"{'learning_rate': 0.1, 'max_depth': 2, 'max_it...",0.3425


Model terbaik berdasarkan MAE validation: HistGradientBoosting
(skill_vs_naive > 0 berarti lebih baik dari Naive; test = 112 jam unik)


In [34]:
fig = go.Figure()
fig.add_scatter(x=wtest.index, y=yte, name="aktual", mode="lines", line=dict(width=3))
fig.add_scatter(x=wtest.index, y=temp_pred["Baseline: Naive (suhu jam ini)"], name="Naive", mode="lines", opacity=0.6)
fig.add_scatter(x=wtest.index, y=temp_pred[temp_best], name=temp_best, mode="lines")
fig.update_layout(title="Suhu 1 jam ke depan pada data test", height=420, xaxis_title="jam target - 1 jam",
                  yaxis_title="°C", hovermode="x unified")
fig.show()

err = pd.DataFrame({"jam": (wtest.index + pd.Timedelta("1h")).hour,
                    "abs_err_naive": np.abs(yte - temp_pred["Baseline: Naive (suhu jam ini)"]),
                    "abs_err_model": np.abs(yte - temp_pred[temp_best])})
display(err.groupby("jam").mean().round(3).T)

jam,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
abs_err_naive,0.4100,0.5180,0.6950,0.3570,0.5720,0.2550,0.8700,2.6100,1.9600,1.5080,0.9240,0.6760,0.4500,0.6820,0.6720,0.4380,1.0220,1.8320,1.0540,0.4620,0.5120,0.4760,1.0130,0.5450
abs_err_model,0.7150,0.6500,0.1690,0.4040,0.3370,0.2430,0.4740,0.5790,0.5810,0.4860,0.4940,0.6500,0.6720,0.7120,0.7580,0.3440,0.7820,0.3770,0.4390,0.7580,0.8150,0.7000,0.8550,0.7410


**Insight B1.6 (suhu).**
- Ketiga model menurunkan MAE 34 sampai 37% dari Naive (0,8768 °C): Linear Regression 0,5541, HistGradientBoosting 0,5765, Random Forest 0,5776. Seasonal Naive (0,9490) lebih buruk dari Naive.
- HistGradientBoosting terpilih lewat MAE validasi (0,5130), tetapi pada test Linear Regression memberi MAE dan RMSE terkecil (0,7268). Selisih antarmodel sekitar 0,02 °C, terlalu kecil untuk menyatakan satu model unggul. Siklus suhu harian yang kuat membuat model linear sudah memadai.
- Error Naive besar pada jam 7 sampai 9 pagi (2,61; 1,96; 1,51 °C) saat suhu naik cepat. Model menekannya menjadi 0,58; 0,58; 0,49 °C. Pada jam 0, 1, 3, 12, 13, dan 14 model sedikit lebih buruk dari Naive.

### B1.7 Eksperimen 2 — Prediksi Kejadian Hujan 1 Jam ke Depan (ML-Q2, ML-Q3)
> **EKSPERIMEN AWAL.** Jumlah kejadian hujan terbatas dan hujan terjadi dalam beberapa episode saja (lihat tabel split), sehingga jumlah sampel efektif jauh lebih kecil daripada jumlah jam. Hasil tidak boleh dibaca sebagai bukti model andal.

**Baseline:** *Persistence* (hujan jam ini → hujan jam berikutnya) dan *Prior* (prevalensi train).
**Model:** Logistic Regression, Random Forest, HistGradientBoosting (class weight seimbang); hyperparameter dipilih pada validation (PR-AUC).
**Metrik:** precision, recall, F1, PR-AUC, ROC-AUC, confusion matrix; threshold klasifikasi 0,5.

In [35]:
def w_clf(y, score, pred=None):
    pred = (np.asarray(score) >= 0.5).astype(int) if pred is None else np.asarray(pred)
    two = len(np.unique(y)) == 2
    return {"precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0),
            "PR_AUC": average_precision_score(y, score) if two else np.nan,
            "ROC_AUC": roc_auc_score(y, score) if two else np.nan}

def w_cm(y, pred):
    return pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1]),
                        index=["aktual: tidak hujan", "aktual: hujan"],
                        columns=["prediksi: tidak hujan", "prediksi: hujan"])

W_CLF_SPECS = {
    "Logistic Regression": (lambda **p: LogisticRegression(class_weight="balanced", max_iter=2000, **p),
                            {"C": [0.1, 1, 10]}),
    "Random Forest": (lambda **p: RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                         random_state=W_RS, n_jobs=-1, **p),
                      {"max_depth": [3, 6, None], "min_samples_leaf": [1, 5]}),
    "HistGradientBoosting": (lambda **p: HistGradientBoostingClassifier(class_weight="balanced", random_state=W_RS, **p),
                             {"learning_rate": [0.05, 0.1], "max_depth": [2, 3], "max_iter": [100, 200]}),
}

if RAIN_READY:
    ytr_c, yva_c, yte_c = wtrain["y_rain"], wval["y_rain"], wtest["y_rain"]
    yfit_c = pd.concat([ytr_c, yva_c])
    rain_rows, rain_models, rain_pred = [], {}, {}

    # Baseline
    pers = wtest["rain_now"].astype(int).values
    rain_rows.append({"model": "Baseline: Persistence", **w_clf(yte_c, pers, pred=pers), "val_PR_AUC": np.nan, "best_params": ""})
    rain_pred["Baseline: Persistence"] = pers
    prior = np.full(len(yte_c), ytr_c.mean())
    rain_rows.append({"model": "Baseline: Prior (prevalensi train)", **w_clf(yte_c, prior), "val_PR_AUC": np.nan, "best_params": ""})

    for name, (mk, grid) in W_CLF_SPECS.items():
        params, val_ap = w_select(mk, grid, Xtr, ytr_c, Xva, yva_c, average_precision_score, True,
                                  lambda m, X: m.predict_proba(X)[:, 1])
        model = w_pipe(mk(**params)).fit(Xfit, yfit_c)
        score = model.predict_proba(Xte)[:, 1]
        rain_models[name], rain_pred[name] = model, score
        rain_rows.append({"model": name, **w_clf(yte_c, score), "val_PR_AUC": val_ap, "best_params": params})

    rain_cmp = pd.DataFrame(rain_rows).set_index("model")
    rain_cmp["prevalensi_test"] = yte_c.mean()
    display(rain_cmp)

    rain_best = max(rain_models, key=lambda m: -np.inf if np.isnan(rain_cmp.loc[m, "val_PR_AUC"]) else rain_cmp.loc[m, "val_PR_AUC"])
    print(f"Model terbaik berdasarkan PR-AUC validation: {rain_best}")
    print(f"Test: {len(yte_c)} jam unik, {int(yte_c.sum())} jam hujan dalam {runs(yte_c)} episode  [EKSPERIMEN AWAL]")
    print("\nConfusion matrix: Persistence"); display(w_cm(yte_c, rain_pred["Baseline: Persistence"]))
    print(f"Confusion matrix: {rain_best}"); display(w_cm(yte_c, (rain_pred[rain_best] >= 0.5).astype(int)))
else:
    print("RAIN_READY = False -> data hujan belum cukup (lihat B1.5); eksperimen hujan dilewati.")

,precision,recall,F1,PR_AUC,ROC_AUC,val_PR_AUC,best_params,prevalensi_test
model,,,,,,,,
Baseline: Persistence,0.8529,0.8529,0.8529,0.7722,0.8944,NaN,,0.3036
Baseline: Prior (prevalensi train),0.0000,0.0000,0.0000,0.3036,0.5000,NaN,,0.3036
Logistic Regression,0.5156,0.9706,0.6735,0.7413,0.9069,0.7980,{'C': 0.1},0.3036
Random Forest,0.8529,0.8529,0.8529,0.8706,0.9284,0.7860,"{'max_depth': None, 'min_samples_leaf': 5}",0.3036
HistGradientBoosting,0.5517,0.9412,0.6957,0.8785,0.9433,0.8070,"{'learning_rate': 0.05, 'max_depth': 2, 'max_i...",0.3036


Model terbaik berdasarkan PR-AUC validation: HistGradientBoosting
Test: 112 jam unik, 34 jam hujan dalam 5 episode  [EKSPERIMEN AWAL]

Confusion matrix: Persistence


,prediksi: tidak hujan,prediksi: hujan
aktual: tidak hujan,73,5
aktual: hujan,5,29


Confusion matrix: HistGradientBoosting


,prediksi: tidak hujan,prediksi: hujan
aktual: tidak hujan,52,26
aktual: hujan,2,32


**Insight B1.7 (hujan, eksperimen awal).**
- Persistence sulit dikalahkan pada ambang 0,5: precision, recall, dan F1 sama-sama 0,853 (73 benar tidak hujan, 5 alarm palsu, 5 terlewat, 29 terdeteksi). Random Forest menyamai skor itu dan unggul pada PR-AUC (0,871 dibanding 0,772).
- HistGradientBoosting punya PR-AUC tertinggi (0,879) dan recall 0,941, dengan hanya 2 dari 34 jam hujan terlewat. Ia memberi 26 alarm palsu, sehingga precision 0,552 dan F1 0,696. Logistic Regression serupa (recall 0,971; precision 0,516).
- Model boosting berguna bila hujan terlewat lebih mahal daripada alarm palsu. Ambang 0,5 belum dioptimalkan.
- Test hanya berisi 5 episode, jadi satu atau dua kejadian mengubah semua metrik. Hasil ini belum menjadi bukti model andal.

### B1.8 Feature Importance (ML-Q4)
Permutation importance pada data test: kenaikan error (suhu: MAE) atau penurunan PR-AUC (hujan) ketika satu fitur diacak. Nilai negatif atau mendekati nol berarti fitur tidak membantu.

In [36]:
imp_t = permutation_importance(temp_models[temp_best], Xte, yte, scoring="neg_mean_absolute_error",
                               n_repeats=30, random_state=W_RS)
imp_temp = (pd.DataFrame({"fitur": W_FEATURES, "kenaikan_MAE": imp_t.importances_mean, "std": imp_t.importances_std})
              .sort_values("kenaikan_MAE", ascending=False).reset_index(drop=True))
display(imp_temp.head(10))
fig = px.bar(imp_temp.sort_values("kenaikan_MAE"), x="kenaikan_MAE", y="fitur", orientation="h", error_x="std",
             title=f"Permutation importance — suhu ({temp_best})")
fig.update_layout(height=640, yaxis_title=None)
fig.show()

,fitur,kenaikan_MAE,std
0,temp_t,0.9896,0.0858
1,temp_yday_target,0.7867,0.0809
2,hour_cos,0.3678,0.0423
3,temp_diff_1,0.1026,0.0202
4,temp_roll6_std,0.0249,0.0095
5,wind_lag_1,0.0115,0.0068
6,precip_roll3_sum,0.0033,0.0088
7,temp_lag_3,0.0029,0.0048
8,precip_lag_1,0.0020,0.0017
9,cloud_lag_1,0.0017,0.0024


In [37]:
if RAIN_READY:
    imp_r = permutation_importance(rain_models[rain_best], Xte, yte_c, scoring="average_precision",
                                   n_repeats=30, random_state=W_RS)
    imp_rain = (pd.DataFrame({"fitur": W_FEATURES, "penurunan_PR_AUC": imp_r.importances_mean, "std": imp_r.importances_std})
                  .sort_values("penurunan_PR_AUC", ascending=False).reset_index(drop=True))
    display(imp_rain.head(10))
    fig = px.bar(imp_rain.sort_values("penurunan_PR_AUC"), x="penurunan_PR_AUC", y="fitur", orientation="h",
                 error_x="std", title=f"Permutation importance — hujan ({rain_best}) [eksperimen awal]")
    fig.update_layout(height=640, yaxis_title=None)
    fig.show()
else:
    print("RAIN_READY = False -> importance hujan dilewati.")

,fitur,penurunan_PR_AUC,std
0,precip_t,0.3793,0.0696
1,humidity_t,0.0507,0.0259
2,rain_hours_24h,0.0102,0.0198
3,temp_diff_1,0.0093,0.0046
4,temp_roll3_mean,0.0078,0.0144
5,hour_cos,0.0078,0.0044
6,cloud_t,0.0048,0.0166
7,temp_roll6_std,0.0045,0.0105
8,temp_lag_1,0.0030,0.0064
9,hour_sin,0.0027,0.0244


**Insight B1.8 (feature importance).**
- Suhu: `temp_t` (kenaikan MAE 0,99), `temp_yday_target` (0,79), `hour_cos` (0,37), dan `temp_diff_1` (0,10) mendominasi, sedangkan fitur lain di bawah 0,03. Prediksi suhu bertumpu pada suhu sekarang, suhu pada jam yang sama kemarin, dan jam dalam sehari.
- Hujan: `precip_t` (penurunan PR-AUC 0,379) jauh di atas `humidity_t` (0,051), dan fitur lain di bawah 0,011. Hujan jam depan terutama ditebak dari hujan jam ini, sejalan dengan kuatnya baseline Persistence. Kelembapan menjadi sinyal tambahan kedua.

### B1.9 Catatan Metodologi & Keterbatasan
- Satu bulan data (Oktober – awal November 2026) dari satu iklim; tidak ada validasi lintas bulan atau musim.
- Cuaca antarruas hampir identik, sehingga 10 ruas tidak menambah informasi independen; sampel efektif ±761 jam.
- ±20 jam terakhir (periode test) berasal dari mode `live`, sedangkan sisanya `backfill`; bila sumbernya berbeda, ada risiko perbedaan karakteristik data pada periode test.
- Sumber API belum dikonfirmasi; jika reanalysis, karakter data berbeda dari observasi stasiun atau prakiraan operasional.
- Hujan terjadi dalam sedikit episode, sehingga metrik hujan bervariasi besar terhadap sedikit kejadian.
- Interpretasi dan kesimpulan ditulis manual setelah hasil ditinjau.

---
### B2. Traffic Congestion Prediction
_Klasifikasi biner · pipeline dan kesiapan data (training menunggu data traffic lebih panjang)_

> Memakai `sv`, `silver_weather`, dan `dim` dari Part A.

### B2.1 Project Overview

**Masalah:** memprediksi apakah suatu ruas jalan akan **macet pada slot 30 menit berikutnya**, berdasarkan informasi yang tersedia sampai waktu prediksi.

**Pertanyaan ML**
1. Bisakah model memprediksi kondisi congestion pada periode berikutnya?
2. Seberapa akurat model mengidentifikasi potensi kemacetan?
3. Fitur apa yang paling berpengaruh terhadap prediksi?

**Keputusan scope**
- Digabung: *Congestion Prediction* + *Traffic Condition Prediction* menjadi satu eksperimen.
- Backlog: *Travel Time Prediction*: waktu tempuh hampir deterministik dari kecepatan × panjang ruas dan data belum punya kolom jarak.
- Target **biner** (macet / tidak macet), bukan 3 kelas, karena distribusi congestion sangat miring (lihat B2.2).
- Ambang "macet" **belum divalidasi**; notebook menampilkan kandidat ambang dan memakai ambang sementara yang ditandai `PROVISIONAL`.

**Prinsip anti-leakage:** fitur pada slot *t* hanya memakai data sampai *t*; target = status macet pada *t + 30 menit*; split berdasarkan waktu (bukan acak) dengan *purge* satu slot di batas train/test.

**Status data saat ini:** hanya ±20 jam, semua akhir pekan, tanpa hujan. Sel pemodelan memiliki **guard kesiapan data** (B2.4): bila positif terlalu sedikit, training dilewati dan notebook hanya menampilkan diagnosis.

In [38]:
import os
import joblib
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ── Konfigurasi Part B ───────────────────────────────────────
RANDOM_STATE = 42
SLOT = pd.Timedelta("30min")
TRAIN_FRAC = 0.70            # porsi waktu untuk train (sisanya test)
MIN_POS_TRAIN = 30           # syarat minimum kelas positif di train
MIN_POS_TEST = 15            # syarat minimum kelas positif di test
MODELS_DIR = Path("models")

### B2.2 Definisi Target (Ambang Macet)

**Referensi:** *Highway Capacity Manual* (HCM 2010), Level of Service (LOS) jalan perkotaan berdasarkan kecepatan perjalanan sebagai persentase dari *base free-flow speed*:

| LOS | Kecepatan / free-flow | Congestion ratio (= 1 − rasio) |
|---|---|---|
| A | > 85% | < 0,15 |
| B | > 67–85% | 0,15 – 0,33 |
| C | > 50–67% | 0,33 – 0,50 |
| D | > 40–50% | 0,50 – 0,60 |
| E | > 30–40% | 0,60 – 0,70 |
| F | ≤ 30% | ≥ 0,70 |

**Catatan penggunaan referensi**
- Batas LOS A di sebagian ringkasan edisi HCM yang lebih baru disebut 80%, bukan 85%; selisihnya relevan untuk ambang 0,15 vs 0,20.
- HCM mengukur kecepatan rata-rata ruas arteri terhadap *base free-flow speed*; di sini `free_flow_speed` dari TomTom dipakai sebagai proksi.
- "Macet" dalam arti LOS biasanya LOS E–F (congestion ≥ 0,60), tetapi **data saat ini tidak punya slot di tingkat itu** (maks ±0,31). Ambang yang memenuhi definisi tersebut menghasilkan nol kelas positif.
- Jadi ambang operasional yang dipakai notebook **bukan definisi final "macet"**, melainkan titik batas yang punya acuan dan masih menghasilkan kelas positif. Keputusan akhir menunggu validasi (data lebih panjang atau acuan lokal Surabaya/Dishub).

In [39]:
# 2.1 Distribusi LOS pada data silver (30 menit)
LOS_BINS = [-np.inf, 0.15, 0.33, 0.50, 0.60, 0.70, np.inf]
LOS_LABELS = list("ABCDEF")
cr = sv["congestion_ratio"].round(6)
sv["los"] = pd.cut(cr, bins=LOS_BINS, labels=LOS_LABELS, right=False)

los_dist = (sv["los"].value_counts().reindex(LOS_LABELS).fillna(0).astype(int)
              .rename("n_slot").to_frame())
los_dist["persen"] = los_dist["n_slot"] / len(sv)
los_dist

,n_slot,persen
los,,
A,502,0.8964
B,56,0.1000
C,2,0.0036
D,0,0.0000
E,0,0.0000
F,0,0.0000


In [40]:
# 2.2 Kandidat ambang: distribusi kelas positif pada tiap kandidat
CANDIDATES = [
    ("> 0 (ada perlambatan)", 0.00, ">"),
    (">= 0.10", 0.10, ">="),
    (">= 0.15 (batas LOS A/B)", 0.15, ">="),
    (">= 0.20", 0.20, ">="),
    (">= 0.25", 0.25, ">="),
    (">= 0.33 (batas LOS B/C)", 0.33, ">="),
    (">= 0.50 (batas LOS C/D)", 0.50, ">="),
    (">= 0.60 (batas LOS D/E)", 0.60, ">="),
    (">= 0.70 (LOS F)", 0.70, ">="),
]

def flag(series, thr, op):
    s = series.round(6)
    return (s > thr) if op == ">" else (s >= thr)

# batas waktu 30% terakhir (calon periode test)
_slots_u = sv["slot"].drop_duplicates().sort_values().reset_index(drop=True)
_cut_preview = _slots_u.iloc[int(len(_slots_u) * TRAIN_FRAC)]

rows = []
for name, thr, op in CANDIDATES:
    f = flag(sv["congestion_ratio"], thr, op)
    rows.append({
        "ambang": name,
        "n_positif": int(f.sum()),
        "persen_positif": f.mean(),
        "ruas_dengan_positif": int(sv.loc[f, "location_id"].nunique()),
        "positif_di_30%_akhir": int((f & (sv["slot"] >= _cut_preview)).sum()),
    })
cand = pd.DataFrame(rows)
display(cand)

fig = px.bar(cand, x="ambang", y="n_positif", text="n_positif",
             title="Jumlah slot positif (macet) per kandidat ambang")
fig.update_layout(height=380, xaxis_title=None)
fig.show()

,ambang,n_positif,persen_positif,ruas_dengan_positif,positif_di_30%_akhir
0,> 0 (ada perlambatan),79,0.1411,8,26
1,>= 0.10,76,0.1357,8,26
2,>= 0.15 (batas LOS A/B),58,0.1036,8,20
3,>= 0.20,45,0.0804,7,17
4,>= 0.25,16,0.0286,5,7
5,>= 0.33 (batas LOS B/C),2,0.0036,1,2
6,>= 0.50 (batas LOS C/D),0,0.0000,0,0
7,>= 0.60 (batas LOS D/E),0,0.0000,0,0
8,>= 0.70 (LOS F),0,0.0000,0,0


In [41]:
# 2.3 Ambang yang dipakai (SEMENTARA)
TARGET_THRESHOLD = 0.15     # batas LOS A/B (HCM 2010): kecepatan < 85% free-flow
TARGET_OP = ">="
TARGET_STATUS = "PROVISIONAL"

n_pos_all = int(flag(sv["congestion_ratio"], TARGET_THRESHOLD, TARGET_OP).sum())
print(f"Target: macet(t+30m) = congestion_ratio {TARGET_OP} {TARGET_THRESHOLD}   [status: {TARGET_STATUS}]")
print(f"Slot positif pada ambang ini: {n_pos_all} dari {len(sv)} ({n_pos_all / len(sv):.1%})")
print("PERHATIAN: definisi target belum divalidasi. Ambang ini = 'tidak lagi arus bebas' (LOS B ke bawah),")
print("           belum tentu 'macet' menurut standar lokal. Hasil model hanya valid untuk definisi ini.")

Target: macet(t+30m) = congestion_ratio >= 0.15   [status: PROVISIONAL]
Slot positif pada ambang ini: 58 dari 560 (10.4%)
PERHATIAN: definisi target belum divalidasi. Ambang ini = 'tidak lagi arus bebas' (LOS B ke bawah),
           belum tentu 'macet' menurut standar lokal. Hasil model hanya valid untuk definisi ini.


**Insight B2.2.** Distribusi LOS pada Silver: A 89,6% (502 slot), B 10,0% (56), C 0,4% (2), D sampai F nol. Definisi macet menurut LOS (E sampai F, congestion ≥ 0,60) menghasilkan nol kelas positif, jadi notebook memakai ambang sementara ≥ 0,15 (batas LOS A/B): 58 positif dari 560 slot (10,4%) di 8 ruas. Label ini berarti "tidak lagi arus bebas" dan belum berarti "macet" menurut standar lokal. Jumlah positif turun cepat bila ambang naik: 79 pada > 0, 45 pada ≥ 0,20, 16 pada ≥ 0,25, dan 2 pada ≥ 0,33.

### B2.3 Feature Engineering (tanpa leakage)

Unit data: **ruas × slot 30 menit**. Semua fitur pada slot *t* memakai data ≤ *t*; target adalah status macet pada slot *t + 1*.

| Kelompok | Fitur |
|---|---|
| Traffic saat ini | `speed_t`, `cong_t`, `is_cong_t` |
| Riwayat | `speed_lag_1`, `speed_lag_2`, `cong_lag_1`, `cong_lag_2`, `speed_diff_1` |
| Rolling (3 slot = 1,5 jam, termasuk *t*) | `speed_roll3_mean`, `cong_roll3_mean`, `cong_roll3_max` |
| Waktu | `hour_of_day`, `minute`, `day_of_week`, `is_weekend` |
| Lokasi | `location_id`, `area`, `frc`, `ffs_road` (free-flow speed ruas) |
| Cuaca (jam yang memuat *t*) | `temperature`, `humidity`, `precipitation`, `wind_speed`, `cloud_cover` |

Slot yang hilang diisi `NaN` (grid 30 menit penuh per ruas) sehingga lag/rolling tidak melompati celah waktu. Fitur konstan di train dibuang otomatis.

In [42]:
def make_panel(sv_df):
    """Grid ruas x slot 30 menit penuh; slot yang hilang -> NaN."""
    base = (sv_df[["location_id", "slot", "current_speed", "congestion_ratio"]]
              .drop_duplicates(["location_id", "slot"]))
    slots = pd.date_range(sv_df["slot"].min(), sv_df["slot"].max(), freq="30min")
    full = pd.MultiIndex.from_product([sorted(sv_df["location_id"].unique()), slots],
                                      names=["location_id", "slot"])
    p = base.set_index(["location_id", "slot"]).reindex(full).reset_index()
    return p.rename(columns={"current_speed": "speed_t", "congestion_ratio": "cong_t"})


def add_features(p, road_static, weather):
    p = p.sort_values(["location_id", "slot"]).reset_index(drop=True)
    g = p.groupby("location_id")
    cong_ok = p["cong_t"].notna()
    p["is_cong_t"] = np.where(cong_ok, flag(p["cong_t"], TARGET_THRESHOLD, TARGET_OP).astype(float), np.nan)

    for k in (1, 2):
        p[f"speed_lag_{k}"] = g["speed_t"].shift(k)
        p[f"cong_lag_{k}"] = g["cong_t"].shift(k)
    p["speed_diff_1"] = p["speed_t"] - p["speed_lag_1"]
    p["speed_roll3_mean"] = g["speed_t"].transform(lambda s: s.rolling(3, min_periods=3).mean())
    p["cong_roll3_mean"] = g["cong_t"].transform(lambda s: s.rolling(3, min_periods=3).mean())
    p["cong_roll3_max"] = g["cong_t"].transform(lambda s: s.rolling(3, min_periods=3).max())

    p["hour_of_day"] = p["slot"].dt.hour
    p["minute"] = p["slot"].dt.minute
    p["day_of_week"] = p["slot"].dt.dayofweek
    p["is_weekend"] = (p["day_of_week"] >= 5).astype(int)

    p = p.merge(road_static, on="location_id", how="left")
    p["hour_ts"] = p["slot"].dt.floor("h")
    w = weather[["location_id", "hour_ts", "temperature", "humidity",
                 "precipitation", "wind_speed", "cloud_cover"]]
    p = p.merge(w, on=["location_id", "hour_ts"], how="left")
    return p


road_static = (sv.groupby("location_id")
                 .agg(frc=("frc", lambda s: s.mode().iat[0]),
                      ffs_road=("free_flow_speed", "median"))
                 .reset_index()
                 .merge(dim[["location_id", "area"]], on="location_id", how="left"))

panel = add_features(make_panel(sv), road_static, silver_weather)
# target: status macet pada slot berikutnya (t + 30 menit) di ruas yang sama
panel["y"] = panel.groupby("location_id")["is_cong_t"].shift(-1)

FEATURE_GROUPS = {
    "traffic_hist": ["speed_t", "cong_t", "is_cong_t", "speed_lag_1", "speed_lag_2",
                     "cong_lag_1", "cong_lag_2", "speed_diff_1",
                     "speed_roll3_mean", "cong_roll3_mean", "cong_roll3_max"],
    "time": ["hour_of_day", "minute", "day_of_week", "is_weekend"],
    "location": ["location_id", "area", "frc", "ffs_road"],
    "weather": ["temperature", "humidity", "precipitation", "wind_speed", "cloud_cover"],
}
CAT_COLS = ["location_id", "area", "frc"]
ALL_FEATURES = [c for grp in FEATURE_GROUPS.values() for c in grp]

print("Panel:", panel.shape, "| slot dengan target terdefinisi:", int(panel["y"].notna().sum()))
panel[["location_id", "slot", "speed_t", "cong_t", "y"] + ["speed_lag_1", "speed_roll3_mean", "temperature"]].head(8)

Panel: (650, 27) | slot dengan target terdefinisi: 550


,location_id,slot,speed_t,cong_t,y,speed_lag_1,speed_roll3_mean,temperature
0,LOC01,2026-10-31 21:00:00+07:00,40.0000,0.0000,NaN,NaN,NaN,29.6000
1,LOC01,2026-10-31 21:30:00+07:00,NaN,NaN,1.0000,40.0000,NaN,29.6000
2,LOC01,2026-10-31 22:00:00+07:00,30.0000,0.2500,NaN,NaN,NaN,28.2000
3,LOC01,2026-10-31 22:30:00+07:00,NaN,NaN,0.0000,30.0000,NaN,28.2000
4,LOC01,2026-10-31 23:00:00+07:00,40.0000,0.0000,NaN,NaN,NaN,27.3000
5,LOC01,2026-10-31 23:30:00+07:00,NaN,NaN,0.0000,40.0000,NaN,27.3000
6,LOC01,2026-11-01 00:00:00+07:00,40.0000,0.0000,0.0000,NaN,NaN,26.7000
7,LOC01,2026-11-01 00:30:00+07:00,40.0000,0.0000,0.0000,40.0000,NaN,26.7000


### B2.4 Temporal Split & Kesiapan Data

- Split berdasarkan **waktu**: `TRAIN_FRAC` pertama = train, sisanya = test (batas waktu yang sama untuk semua ruas).
- **Purge:** baris train yang slot targetnya (*t + 30 menit*) sudah masuk periode test dibuang, agar label train tidak memuat informasi dari periode test.
- Guard: training hanya jalan bila kelas positif cukup (`MIN_POS_TRAIN`, `MIN_POS_TEST`).

In [43]:
req = [c for c in ALL_FEATURES if c not in CAT_COLS] + CAT_COLS
data = panel.dropna(subset=["y"] + req).sort_values(["slot", "location_id"]).reset_index(drop=True)
data["y"] = data["y"].astype(int)

slots_u = data["slot"].drop_duplicates().sort_values().reset_index(drop=True)
cut = slots_u.iloc[int(len(slots_u) * TRAIN_FRAC)]
train = data[data["slot"] + SLOT < cut].copy()      # purge: slot target < cut
test = data[data["slot"] >= cut].copy()

status = pd.DataFrame({
    "baris": [len(train), len(test)],
    "positif": [int(train["y"].sum()), int(test["y"].sum())],
    "prevalensi": [train["y"].mean(), test["y"].mean()] if len(train) and len(test) else [np.nan, np.nan],
    "mulai": [train["slot"].min(), test["slot"].min()],
    "akhir": [train["slot"].max(), test["slot"].max()],
}, index=["train", "test"])
display(status)
print(f"Batas split (cut): {cut}  | baris dibuang karena purge/NaN: {len(panel) - len(train) - len(test)}")

READY = (int(train["y"].sum()) >= MIN_POS_TRAIN and int(test["y"].sum()) >= MIN_POS_TEST
         and train["y"].nunique() == 2 and test["y"].nunique() == 2)
if READY:
    print("\nREADY = True -> training dijalankan.")
else:
    print(f"\nREADY = False -> data belum cukup: butuh positif train >= {MIN_POS_TRAIN} dan test >= {MIN_POS_TEST}.")
    print("Training/evaluasi dilewati. Tambah data (>= 2-4 minggu, termasuk hari kerja & hujan) atau validasi ulang ambang target.")

,baris,positif,prevalensi,mulai,akhir
train,310,27,0.0871,2026-11-01 01:00:00+07:00,2026-11-01 16:00:00+07:00
test,150,24,0.1600,2026-11-01 17:00:00+07:00,2026-11-02 04:30:00+07:00


Batas split (cut): 2026-11-01 17:00:00+07:00  | baris dibuang karena purge/NaN: 190

READY = False -> data belum cukup: butuh positif train >= 30 dan test >= 15.
Training/evaluasi dilewati. Tambah data (>= 2-4 minggu, termasuk hari kerja & hujan) atau validasi ulang ambang target.


**Insight B2.4 (kesiapan data).** Setelah purge dan pembuangan NaN, 190 baris keluar dan tersisa 460 baris berlabel. Train berisi 310 baris dengan 27 positif (8,7%; 1 Nov 01:00 sampai 16:00). Test berisi 150 baris dengan 24 positif (16,0%; 1 Nov 17:00 sampai 2 Nov 04:30). Test sudah melewati syarat minimum 15 positif, tetapi train kurang 3 positif dari syarat 30. Hasilnya `READY = False`, sehingga B2.5 sampai B2.8 (perbandingan model, ablation, importance, penyimpanan model) dilewati. Pipeline, fitur, split, dan fungsi inference sudah siap, tetapi belum ada ukuran akurasi untuk prediksi kemacetan. Test mencakup jam 17 sampai 19 yang termasuk puncak (Q2), dan prevalensinya hampir dua kali prevalensi train.

### B2.5 Baseline & Perbandingan Model

**Baseline:** (1) *Persistence*: status macet sekarang = prediksi slot berikutnya; (2) *Historical average*: rata-rata kelas positif per ruas × jam dari data train.
**Model:** Logistic Regression, Random Forest, Histogram Gradient Boosting (sklearn; bisa diganti XGBoost/LightGBM bila terpasang).
**Tuning:** `GridSearchCV` + `TimeSeriesSplit` (scoring PR-AUC). Model final dipilih berdasarkan skor CV, bukan skor test.
**Metrik:** precision, recall, F1, PR-AUC, ROC-AUC (bukan hanya accuracy karena kelas tidak seimbang). Threshold klasifikasi tetap 0,5.

In [44]:
def usable(cols, frame):
    """Buang fitur konstan di train."""
    return [c for c in cols if frame[c].nunique(dropna=True) > 1]

def build_pipe(est, cols):
    cat = [c for c in cols if c in CAT_COLS]
    num = [c for c in cols if c not in CAT_COLS]
    pre = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
    ])
    return Pipeline([("pre", pre), ("clf", est)])

def evaluate(name, y_true, score, pred=None):
    pred = (np.asarray(score) >= 0.5).astype(int) if pred is None else np.asarray(pred)
    two = len(np.unique(y_true)) == 2
    return {
        "model": name,
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, score) if two else np.nan,
        "roc_auc": roc_auc_score(y_true, score) if two else np.nan,
    }

def fit_tuned(pipe, grid, X, y):
    cv = TimeSeriesSplit(n_splits=3)
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="average_precision", error_score=np.nan, refit=True)
    try:
        gs.fit(X, y)
        return gs.best_estimator_, gs.best_params_, float(gs.best_score_)
    except Exception as e:                       # fold tanpa positif, dll.
        print(f"  tuning gagal ({type(e).__name__}); pakai parameter default")
        return pipe.fit(X, y), {}, np.nan

In [45]:
results, fitted = [], {}

if READY:
    feats = usable(ALL_FEATURES, train)
    Xtr, ytr = train[feats], train["y"]
    Xte, yte = test[feats], test["y"]
    print(f"Fitur terpakai ({len(feats)}): {feats}\n")

    # Baseline 1: persistence (status sekarang -> slot berikutnya)
    results.append(evaluate("Baseline: Persistence", yte, test["cong_t"].fillna(0).values,
                            pred=test["is_cong_t"].astype(int).values))

    # Baseline 2: historical average ruas x jam
    rate = train.groupby(["location_id", "hour_of_day"])["y"].mean().rename("rate").reset_index()
    te_rate = test.merge(rate, on=["location_id", "hour_of_day"], how="left")["rate"].fillna(ytr.mean()).values
    results.append(evaluate("Baseline: Historical Average", yte, te_rate))

    # Model + grid
    specs = {
        "Logistic Regression": (LogisticRegression(class_weight="balanced", max_iter=2000),
                                {"clf__C": [0.1, 1, 10]}),
        "Random Forest": (RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                 random_state=RANDOM_STATE, n_jobs=-1),
                          {"clf__max_depth": [3, 5, None], "clf__min_samples_leaf": [1, 3]}),
        "HistGradientBoosting": (HistGradientBoostingClassifier(class_weight="balanced",
                                                                random_state=RANDOM_STATE),
                                 {"clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3],
                                  "clf__max_iter": [100, 200]}),
    }
    cv_scores = {}
    for name, (est, grid) in specs.items():
        print("Tuning:", name)
        best, params, cvs = fit_tuned(build_pipe(est, feats), grid, Xtr, ytr)
        fitted[name], cv_scores[name] = best, cvs
        r = evaluate(name, yte, best.predict_proba(Xte)[:, 1])
        r.update({"cv_pr_auc": cvs, "best_params": params})
        results.append(r)

    comp = pd.DataFrame(results).set_index("model")
    comp["prevalensi_test"] = yte.mean()
    display(comp)

    final_name = max(cv_scores, key=lambda k: -np.inf if np.isnan(cv_scores[k]) else cv_scores[k])
    final = fitted[final_name]
    print(f"\nModel final (skor CV tertinggi): {final_name}")
else:
    print("READY = False -> perbandingan model dilewati (lihat B2.4).")

READY = False -> perbandingan model dilewati (lihat B2.4).


In [46]:
if READY:
    p = final.predict_proba(Xte)[:, 1]
    cm = pd.DataFrame(confusion_matrix(yte, (p >= 0.5).astype(int), labels=[0, 1]),
                      index=["aktual: tidak macet", "aktual: macet"],
                      columns=["prediksi: tidak macet", "prediksi: macet"])
    display(cm)

    comp_plot = comp.reset_index().melt(id_vars="model", value_vars=["precision", "recall", "f1", "pr_auc"])
    fig = px.bar(comp_plot, x="model", y="value", color="variable", barmode="group",
                 title="Perbandingan model pada data test")
    fig.update_layout(height=420, xaxis_title=None)
    fig.show()

### B2.6 Ablation: Peran Riwayat Traffic vs Cuaca
Model final (jenis & hyperparameter sama) dilatih ulang dengan kelompok fitur berbeda. Pada data saat ini cuaca praktis tidak bervariasi (tanpa hujan), jadi hasil ablation baru bermakna setelah data bertambah.

In [47]:
if READY:
    base_est = clone(final.named_steps["clf"])
    sets = {
        "time + location": FEATURE_GROUPS["time"] + FEATURE_GROUPS["location"],
        "weather + time + location (tanpa riwayat traffic)": FEATURE_GROUPS["weather"] + FEATURE_GROUPS["time"] + FEATURE_GROUPS["location"],
        "traffic history + time + location": FEATURE_GROUPS["traffic_hist"] + FEATURE_GROUPS["time"] + FEATURE_GROUPS["location"],
        "semua fitur (+ cuaca)": ALL_FEATURES,
    }
    rows = []
    for name, cols in sets.items():
        cols = usable(cols, train)
        pipe = build_pipe(clone(base_est), cols).fit(train[cols], ytr)
        r = evaluate(name, yte, pipe.predict_proba(test[cols])[:, 1])
        r["n_fitur"] = len(cols)
        rows.append(r)
    display(pd.DataFrame(rows).set_index("model"))
else:
    print("READY = False -> ablation dilewati.")

READY = False -> ablation dilewati.


### B2.7 Feature Importance
Permutation importance pada data test (penurunan PR-AUC saat fitur diacak). Dihitung pada kolom input asli, sehingga fitur kategorikal tidak terpecah per kategori.

In [48]:
if READY:
    pi = permutation_importance(final, Xte, yte, scoring="average_precision",
                                n_repeats=20, random_state=RANDOM_STATE)
    imp = (pd.DataFrame({"fitur": feats, "importance": pi.importances_mean, "std": pi.importances_std})
             .sort_values("importance", ascending=False).reset_index(drop=True))
    display(imp)
    fig = px.bar(imp.sort_values("importance"), x="importance", y="fitur", orientation="h",
                 error_x="std", title=f"Permutation importance ({final_name})")
    fig.update_layout(height=max(380, 24 * len(imp)), yaxis_title=None)
    fig.show()
else:
    print("READY = False -> feature importance dilewati.")

READY = False -> feature importance dilewati.


### B2.8 Inference & Penyimpanan Model
`predict_next_slot` membangun fitur dari riwayat terbaru dan menghasilkan probabilitas macet 30 menit ke depan per ruas. Fungsi yang sama dapat dibungkus FastAPI pada tahap deployment (file terpisah di luar notebook).

In [49]:
def predict_next_slot(bundle, sv_recent, weather, road_static):
    """Probabilitas macet pada slot berikutnya untuk tiap ruas, dari data silver terbaru."""
    p = add_features(make_panel(sv_recent), road_static, weather)
    latest = p.sort_values("slot").groupby("location_id").tail(1)
    cols = bundle["features"]
    latest = latest.dropna(subset=[c for c in cols if c not in CAT_COLS])
    out = latest[["location_id", "slot"]].copy()
    out["slot_prediksi"] = out["slot"] + SLOT
    out["p_macet"] = bundle["pipeline"].predict_proba(latest[cols])[:, 1]
    return out.merge(dim[["location_id", "location_name"]], on="location_id")

if READY:
    MODELS_DIR.mkdir(exist_ok=True)
    bundle = {"pipeline": final, "features": feats, "model": final_name,
              "target": {"threshold": TARGET_THRESHOLD, "op": TARGET_OP, "status": TARGET_STATUS,
                         "horizon": "30min"}}
    joblib.dump(bundle, MODELS_DIR / "congestion_clf.joblib")
    print("Model disimpan:", (MODELS_DIR / "congestion_clf.joblib").resolve())
    display(predict_next_slot(bundle, sv, silver_weather, road_static)
              .sort_values("p_macet", ascending=False)
              [["location_name", "slot", "slot_prediksi", "p_macet"]].reset_index(drop=True))
else:
    print("READY = False -> model belum disimpan; fungsi inference sudah terdefinisi dan siap dipakai setelah training.")

READY = False -> model belum disimpan; fungsi inference sudah terdefinisi dan siap dipakai setelah training.


### B2.9 Backlog & Syarat Data Minimum

**Syarat sebelum hasil model dapat diinterpretasikan**
- Data traffic ≥ 2–4 minggu (idealnya mencakup hari kerja dan akhir pekan).
- Ada hari hujan agar peran cuaca bisa diuji.
- Ambang "macet" divalidasi (acuan lokal atau distribusi data yang lebih panjang); setelah itu ubah `TARGET_THRESHOLD`, `TARGET_OP`, dan `TARGET_STATUS` di B2.2.
- Kelas positif memenuhi `MIN_POS_TRAIN` dan `MIN_POS_TEST`.

**Backlog**
- Travel Time Prediction (butuh target travel time berkualitas dan atribut jarak ruas).
- Deployment API (FastAPI) membungkus `predict_next_slot` dan `congestion_clf.joblib`.
- Opsional: SHAP, XGBoost/LightGBM, kalibrasi probabilitas, pemilihan threshold klasifikasi pada validation set.